# Pravaah: train the forecast models

Trains the gradient-boosted **wet-bulb** and **carbon** models for every region, judges each against two
baselines (persistence and the provider's own forecast) on the last 7 days, and keeps a model **only if it
beats both**. The kept models are tiny JSON files that the deployed `predict` Lambda loads.

**Run all** (Runtime → Run all). It takes a few minutes. At the end it downloads `forecast-models.zip`.

What you need:
- Internet (Colab has it). Weather comes from Open-Meteo's free API, which has a daily request quota per
  IP address. If a cell fails with *429 / Daily API request limit exceeded*, wait until 00:00 UTC and rerun.
- *Optional, for carbon models:* real carbon history. The deployed pipeline saves it to S3; carbon models are
  skipped until at least 10 days exist. See the upload cell below. Weather models need nothing extra.

In [ ]:
!pip -q install scikit-learn pyyaml pandas

In [ ]:
#@title Unpack the project files (embedded; nothing to edit)
import base64, pathlib

FILES = {
"data/regions.yaml": "IyBBV1MgcmVnaW9ucyBQcmF2YWFoIGNhbiBwbGFjZSBqb2JzIGluLgojCiMgU3RlcCAxIG5lZWRzOiBsb2NhdGlvbiAoZm9yIHdlYXRoZXIpLCBjb29saW5nIHR5cGUsIGFuZCB3aGljaCBkaXNjbG9zZWQgV1VFIHRvCiMgY2FsaWJyYXRlIGFnYWluc3QuIFN0ZXAgMiBmaWxscyBpbiB0aGUgcmVzdCBmcm9tIHJlYWwgZGF0YToKIyAgIC0gZWxlY3RyaWNpdHlfbWFwc196b25lOiBjb25maXJtIGVhY2ggYWdhaW5zdCB0aGUgRWxlY3RyaWNpdHkgTWFwcyB6b25lIGxpc3QKIyAgIC0gZ3JpZF9taXggLyB0eXBpY2FsX2NpX2dfcGVyX2t3aDogUExBQ0VIT0xERVJTIChyb3VnaCBhbm51YWwgc2hhcmVzIGFuZAojICAgICBpbnRlbnNpdGllcyBmcm9tIG1lbW9yeSkgc28gdGhlIENMSSBydW5zIGJlZm9yZSB0aGUgcGlwZWxpbmUgZXhpc3RzLgojICAgICBSZXBsYWNlIHdpdGggRWxlY3RyaWNpdHkgTWFwcyBwb3dlci1icmVha2Rvd24gaGlzdG9yeS4KIyAgIC0gd2F0ZXJfc3RyZXNzX3Njb3JlOiBXUkkgQXF1ZWR1Y3QgYmFzZWxpbmUgd2F0ZXIgc3RyZXNzICgwLTUpIGZvciB0aGUgc2l0ZSdzCiMgICAgIGJhc2luOyBudWxsIG1lYW5zICJub3QgbG9va2VkIHVwIHlldCIgYW5kIGdpdmVzIGEgbXVsdGlwbGllciBvZiAxLjAuCiMgICAtIHd1ZV9zY2FsZTogd3JpdHRlbiBieSBzY3JpcHRzL2NhbGlicmF0ZV93dWUucHkgZnJvbSBhIHllYXIgb2Ygd2VhdGhlciBoaXN0b3J5LgojCiMgQ29vcmRpbmF0ZXMgYXJlIHRoZSBtZXRybyBhcmVhIHRoZSByZWdpb24gaXMgbmFtZWQgYWZ0ZXIsIG5vdCBmYWNpbGl0eSBzaXRlcwojIChBV1MgZG9lcyBub3QgcHVibGlzaCB0aG9zZSkuCiMKIyBnZW86IGxlZ2FsIGp1cmlzZGljdGlvbiBmb3IgZGF0YSByZXNpZGVuY3kgKGpvYnMgZmxhZ2dlZCByZXNpZGVudCBzdGF5IGluIHRoZWlyCiMgc3VibWl0IHJlZ2lvbidzIGdlbykuIEVVIGNvdmVycyB0aGUgRVUvRUVBIHJlZ2lvbnMuCiMKIyBDb29saW5nIHR5cGVzOiBBV1MgZGVzY3JpYmVzIGRpcmVjdCBldmFwb3JhdGl2ZSAoImFkaWFiYXRpYyIpIGNvb2xpbmcgZm9yIG1vc3QKIyByZWdpb25zOyBpbiBob3QgaHVtaWQgY2xpbWF0ZXMgd2hlcmUgdGhhdCBjYW5ub3Qgd29yayAod2V0LWJ1bGIgbmVhciBzZXRwb2ludAojIGFsbCB5ZWFyKSB3ZSBhc3N1bWUgY2hpbGxlZCB3YXRlciB3aXRoIGNvb2xpbmcgdG93ZXJzLgoKcmVnaW9uczoKICAtIGlkOiBhcC1zb3V0aC0xCiAgICBuYW1lOiBBc2lhIFBhY2lmaWMgKE11bWJhaSkKICAgIGdlbzogSU4KICAgIGxhdDogMTkuMDgKICAgIGxvbjogNzIuODgKICAgIGNvb2xpbmdfdHlwZTogdG93ZXIKICAgIGVsZWN0cmljaXR5X21hcHNfem9uZTogSU4tV0UKICAgIGRpc2Nsb3NlZF93dWVfa2V5OiBhc2lhLXBhY2lmaWMtYWdncmVnYXRlICAgIyBubyBNdW1iYWktc3BlY2lmaWMgZmlndXJlCiAgICB3YXRlcl9zdHJlc3Nfc2NvcmU6IG51bGwKICAgIGdyaWRfbWl4OiB7cGxhY2Vob2xkZXI6IHRydWUsIHNoYXJlczoge2NvYWw6IDAuNzIsIGdhczogMC4wMywgaHlkcm86IDAuMDgsIG51Y2xlYXI6IDAuMDMsIHNvbGFyOiAwLjA4LCB3aW5kOiAwLjA1LCBiaW9tYXNzOiAwLjAxfX0KICAgIHR5cGljYWxfY2lfZ19wZXJfa3doOiA2ODAgICAgICAgICAgICAjIHBsYWNlaG9sZGVyCgogIC0gaWQ6IGFwLXNvdXRoLTIKICAgIG5hbWU6IEFzaWEgUGFjaWZpYyAoSHlkZXJhYmFkKQogICAgZ2VvOiBJTgogICAgbGF0OiAxNy4zOQogICAgbG9uOiA3OC40OQogICAgY29vbGluZ190eXBlOiB0b3dlcgogICAgZWxlY3RyaWNpdHlfbWFwc196b25lOiBJTi1TTwogICAgZGlzY2xvc2VkX3d1ZV9rZXk6IGFzaWEtcGFjaWZpYy1hZ2dyZWdhdGUKICAgIHdhdGVyX3N0cmVzc19zY29yZTogbnVsbAogICAgZ3JpZF9taXg6IHtwbGFjZWhvbGRlcjogdHJ1ZSwgc2hhcmVzOiB7Y29hbDogMC42MiwgZ2FzOiAwLjAyLCBoeWRybzogMC4wOCwgbnVjbGVhcjogMC4wNCwgc29sYXI6IDAuMTMsIHdpbmQ6IDAuMTAsIGJpb21hc3M6IDAuMDF9fQogICAgdHlwaWNhbF9jaV9nX3Blcl9rd2g6IDYwMCAgICAgICAgICAgICMgcGxhY2Vob2xkZXIKCiAgLSBpZDogYXAtc291dGhlYXN0LTEKICAgIG5hbWU6IEFzaWEgUGFjaWZpYyAoU2luZ2Fwb3JlKQogICAgZ2VvOiBTRwogICAgbGF0OiAxLjM1CiAgICBsb246IDEwMy44MgogICAgY29vbGluZ190eXBlOiB0b3dlcgogICAgZWxlY3RyaWNpdHlfbWFwc196b25lOiBTRwogICAgd2F0ZXJfc3RyZXNzX3Njb3JlOiBudWxsCiAgICBncmlkX21peDoge3BsYWNlaG9sZGVyOiB0cnVlLCBzaGFyZXM6IHtnYXM6IDAuOTQsIG9pbDogMC4wMSwgc29sYXI6IDAuMDMsIGJpb21hc3M6IDAuMDJ9fQogICAgdHlwaWNhbF9jaV9nX3Blcl9rd2g6IDQ4MCAgICAgICAgICAgICMgcGxhY2Vob2xkZXIKCiAgLSBpZDogZXUtbm9ydGgtMQogICAgbmFtZTogRXVyb3BlIChTdG9ja2hvbG0pCiAgICBnZW86IEVVCiAgICBsYXQ6IDU5LjMzCiAgICBsb246IDE4LjA3CiAgICBjb29saW5nX3R5cGU6IGFkaWFiYXRpYwogICAgZWxlY3RyaWNpdHlfbWFwc196b25lOiBTRS1TRTMKICAgIHdhdGVyX3N0cmVzc19zY29yZTogbnVsbAogICAgZ3JpZF9taXg6IHtwbGFjZWhvbGRlcjogdHJ1ZSwgc2hhcmVzOiB7aHlkcm86IDAuNDAsIG51Y2xlYXI6IDAuMzAsIHdpbmQ6IDAuMjAsIGJpb21hc3M6IDAuMDcsIHNvbGFyOiAwLjAyLCBnYXM6IDAuMDF9fQogICAgdHlwaWNhbF9jaV9nX3Blcl9rd2g6IDM1ICAgICAgICAgICAgICMgcGxhY2Vob2xkZXIKCiAgLSBpZDogZXUtd2VzdC0xCiAgICBuYW1lOiBFdXJvcGUgKElyZWxhbmQpCiAgICBnZW86IEVVCiAgICBsYXQ6IDUzLjM1CiAgICBsb246IC02LjI2CiAgICBjb29saW5nX3R5cGU6IGFkaWFiYXRpYwogICAgZWxlY3RyaWNpdHlfbWFwc196b25lOiBJRQogICAgd2F0ZXJfc3RyZXNzX3Njb3JlOiBudWxsCiAgICBncmlkX21peDoge3BsYWNlaG9sZGVyOiB0cnVlLCBzaGFyZXM6IHtnYXM6IDAuNDUsIHdpbmQ6IDAuMzUsIGNvYWw6IDAuMDMsIGh5ZHJvOiAwLjAyLCBzb2xhcjogMC4wMywgYmlvbWFzczogMC4wMiwgdW5rbm93bjogMC4xMH19CiAgICB0eXBpY2FsX2NpX2dfcGVyX2t3aDogMzAwICAgICAgICAgICAgIyBwbGFjZWhvbGRlcgoKICAtIGlkOiBldS1jZW50cmFsLTEKICAgIG5hbWU6IEV1cm9wZSAoRnJhbmtmdXJ0KQogICAgZ2VvOiBFVQogICAgbGF0OiA1MC4xMQogICAgbG9uOiA4LjY4CiAgICBjb29saW5nX3R5cGU6IGFkaWFiYXRpYwogICAgZWxlY3RyaWNpdHlfbWFwc196b25lOiBERQogICAgd2F0ZXJfc3RyZXNzX3Njb3JlOiBudWxsCiAgICBncmlkX21peDoge3BsYWNlaG9sZGVyOiB0cnVlLCBzaGFyZXM6IHt3aW5kOiAwLjMyLCBjb2FsOiAwLjIyLCBzb2xhcjogMC4xMywgZ2FzOiAwLjE1LCBiaW9tYXNzOiAwLjA4LCBoeWRybzogMC4wNCwgdW5rbm93bjogMC4wNn19CiAgICB0eXBpY2FsX2NpX2dfcGVyX2t3aDogMzgwICAgICAgICAgICAgIyBwbGFjZWhvbGRlcgoKICAtIGlkOiB1cy1lYXN0LTEKICAgIG5hbWU6IFVTIEVhc3QgKE4uIFZpcmdpbmlhKQogICAgZ2VvOiBVUwogICAgbGF0OiAzOS4wNAogICAgbG9uOiAtNzcuNDkKICAgIGNvb2xpbmdfdHlwZTogYWRpYWJhdGljCiAgICBlbGVjdHJpY2l0eV9tYXBzX3pvbmU6IFVTLU1JREEtUEpNCiAgICB3YXRlcl9zdHJlc3Nfc2NvcmU6IG51bGwKICAgIGdyaWRfbWl4OiB7cGxhY2Vob2xkZXI6IHRydWUsIHNoYXJlczoge2dhczogMC40MiwgbnVjbGVhcjogMC4zMywgY29hbDogMC4xNSwgd2luZDogMC4wNCwgc29sYXI6IDAuMDMsIGh5ZHJvOiAwLjAyLCBvaWw6IDAuMDF9fQogICAgdHlwaWNhbF9jaV9nX3Blcl9rd2g6IDM4MCAgICAgICAgICAgICMgcGxhY2Vob2xkZXIKCiAgLSBpZDogdXMtd2VzdC0yCiAgICBuYW1lOiBVUyBXZXN0IChPcmVnb24pCiAgICBnZW86IFVTCiAgICBsYXQ6IDQ1Ljg0CiAgICBsb246IC0xMTkuNzAKICAgIGNvb2xpbmdfdHlwZTogYWRpYWJhdGljCiAgICBlbGVjdHJpY2l0eV9tYXBzX3pvbmU6IFVTLU5XLUJQQVQKICAgIHdhdGVyX3N0cmVzc19zY29yZTogbnVsbAogICAgZ3JpZF9taXg6IHtwbGFjZWhvbGRlcjogdHJ1ZSwgc2hhcmVzOiB7aHlkcm86IDAuNzUsIG51Y2xlYXI6IDAuMDgsIHdpbmQ6IDAuMTAsIGdhczogMC4wNiwgc29sYXI6IDAuMDF9fQogICAgdHlwaWNhbF9jaV9nX3Blcl9rd2g6IDEwMCAgICAgICAgICAgICMgcGxhY2Vob2xkZXIK",
"scripts/train_forecasts.py": "IiIiU3RlcCA0OiB0cmFpbiBhbmQganVkZ2UgdGhlIHdldC1idWxiIGFuZCBjYXJib24gbW9kZWxzIHBlciByZWdpb24uCgogICAgcGlwIGluc3RhbGwgLXIgcmVxdWlyZW1lbnRzLWRldi50eHQgICAgICMgaW5jbHVkZXMgc2Npa2l0LWxlYXJuCiAgICBweXRob24gc2NyaXB0cy90cmFpbl9mb3JlY2FzdHMucHkgLS1kYXlzIDYwCgpXZXQtYnVsYjogT3Blbi1NZXRlbydzIEhpc3RvcmljYWwgRm9yZWNhc3QgQVBJICh3aGF0IHRoZSBmb3JlY2FzdCBzYWlkKSBhZ2FpbnN0IHRoZQphcmNoaXZlICh3aGF0IGhhcHBlbmVkKSwgbGFzdCBOIGRheXM7IHBlcnNpc3RlbmNlID0gb2JzZXJ2ZWQgd2V0LWJ1bGIgMjQgaCBlYXJsaWVyLgpUaGUgYXJjaGl2ZWQgZm9yZWNhc3QgaXMgc3RpdGNoZWQgZnJvbSBzaG9ydCBsZWFkcywgc28gdGhpcyBsZWFybnMgdGhlIHByb3ZpZGVyJ3MKc3lzdGVtYXRpYyBiaWFzIGJ5IGhvdXIgYW5kIHNlYXNvbiByYXRoZXIgdGhhbiBpdHMgZXJyb3IgZ3Jvd3RoIHdpdGggbGVhZCB0aW1lLgoKQ2FyYm9uOiByZWFsIChub24tbW9kZWxsZWQpIHJvd3MgaW4gZGF0YS9oaXN0b3J5LzxyZWdpb24+LmNzdi4gVGhlIGZyZWUgRWxlY3RyaWNpdHkKTWFwcyB0aWVyIGdpdmVzIDI0IGgsIHNvIGNhcmJvbiBtb2RlbHMgbmVlZCBoaXN0b3J5IGFjY3VtdWxhdGVkIGJ5IHRoZSBkZXBsb3llZApwaXBlbGluZSBmaXJzdDsgcmVnaW9ucyB3aXRob3V0IGVub3VnaCBhcmUgc2tpcHBlZCBhbmQgc2F5IHNvLgoKRWFjaCBtb2RlbCBpcyBrZXB0IG9ubHkgaWYgaXRzIGhlbGQtb3V0IE1BRSAobGFzdCA3IGRheXMpIGJlYXRzIHBlcnNpc3RlbmNlIGFuZCB0aGUKcHJvdmlkZXIuIFdyaXRlcyBmb3JlY2FzdGluZy9tb2RlbHMvPHJlZ2lvbj4tPHRhcmdldD4uanNvbiAod2l0aCB0aGUgdmVyZGljdCkgYW5kCmZvcmVjYXN0aW5nL21vZGVscy9tZXRyaWNzLmpzb24uIFRoZSBwcmVkaWN0IExhbWJkYSBsb2FkcyBvbmx5IHVzZV9tb2RlbD10cnVlIGZpbGVzLgoiIiIKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY3N2CmltcG9ydCBqc29uCmltcG9ydCBzeXMKZnJvbSBkYXRldGltZSBpbXBvcnQgZGF0ZXRpbWUsIHRpbWVkZWx0YSwgdGltZXpvbmUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudC5wYXJlbnQKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihST09UKSkKCmZyb20gZm9yZWNhc3RpbmcuZXZhbHVhdGUgaW1wb3J0IGZpdF9hbmRfanVkZ2UgICMgbm9xYTogRTQwMgpmcm9tIGZvcmVjYXN0aW5nLmZlYXR1cmVzIGltcG9ydCBDQVJCT05fRkVBVFVSRVMsIFdFVEJVTEJfRkVBVFVSRVMsIGNhcmJvbl9yb3csIHdldGJ1bGJfcm93ICAjIG5vcWE6IEU0MDIKZnJvbSBtb2RlbCBpbXBvcnQgcmVnaW9ucyBhcyByZWdpb25fY29uZmlnICAjIG5vcWE6IEU0MDIKZnJvbSBtb2RlbC53ZXRidWxiIGltcG9ydCB3ZXRfYnVsYiAgIyBub3FhOiBFNDAyCmZyb20gc291cmNlcyBpbXBvcnQgb3Blbm1ldGVvICAjIG5vcWE6IEU0MDIKCk9VVCA9IFJPT1QgLyAiZm9yZWNhc3RpbmciIC8gIm1vZGVscyIKSElTVF9GQyA9ICJodHRwczovL2hpc3RvcmljYWwtZm9yZWNhc3QtYXBpLm9wZW4tbWV0ZW8uY29tL3YxL2ZvcmVjYXN0IgpURVNUX0RBWVMgPSA3CgoKZGVmIHNlcmllcyh1cmwsIHJlZ2lvbiwgc3RhcnQsIGVuZCkgLT4gZGljdDoKICAgIHJhdyA9IG9wZW5tZXRlby5fZ2V0KHVybCwgeyJsYXRpdHVkZSI6IHJlZ2lvbi5sYXQsICJsb25naXR1ZGUiOiByZWdpb24ubG9uLCAiaG91cmx5IjogIiwiLmpvaW4ob3Blbm1ldGVvLlZBUlMpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgInRpbWV6b25lIjogIlVUQyIsICJzdGFydF9kYXRlIjogZiJ7c3RhcnQ6JVktJW0tJWR9IiwgImVuZF9kYXRlIjogZiJ7ZW5kOiVZLSVtLSVkfSJ9LAogICAgICAgICAgICAgICAgICAgICAgICAgdGltZW91dD02MC4wKQogICAgaCA9IHJhd1siaG91cmx5Il0KICAgIHJldHVybiB7dDogKFQsIFJILCBQKSBmb3IgdCwgVCwgUkgsIFAgaW4gemlwKGhbInRpbWUiXSwgaFsidGVtcGVyYXR1cmVfMm0iXSwgaFsicmVsYXRpdmVfaHVtaWRpdHlfMm0iXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaFsic3VyZmFjZV9wcmVzc3VyZSJdKSBpZiBOb25lIG5vdCBpbiAoVCwgUkgsIFApfQoKCmRlZiB3ZXRidWxiX2RhdGFzZXQocmVnaW9uLCBkYXlzOiBpbnQpOgogICAgZW5kID0gZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykucmVwbGFjZSh0emluZm89Tm9uZSkgLSB0aW1lZGVsdGEoZGF5cz02KQogICAgc3RhcnQgPSBlbmQgLSB0aW1lZGVsdGEoZGF5cz1kYXlzKQogICAgZmMgPSBzZXJpZXMoSElTVF9GQywgcmVnaW9uLCBzdGFydCwgZW5kKQogICAgb2JzID0gc2VyaWVzKG9wZW5tZXRlby5BUkNISVZFX1VSTCwgcmVnaW9uLCBzdGFydCwgZW5kKQogICAgaG91cnMgPSBzb3J0ZWQoc2V0KGZjKSAmIHNldChvYnMpKQogICAgb2JzX3diID0ge3Q6IHdldF9idWxiKFQsIG1heChSSCwgMSksIFApIGZvciB0LCAoVCwgUkgsIFApIGluIG9icy5pdGVtcygpfQogICAgZmNfd2IgPSB7dDogd2V0X2J1bGIoVCwgbWF4KFJILCAxKSwgUCkgZm9yIHQsIChULCBSSCwgUCkgaW4gZmMuaXRlbXMoKX0KICAgIHJvd3MsIHksIHBlcnMsIHByb3YsIHRlc3QgPSBbXSwgW10sIFtdLCBbXSwgW10KICAgIGN1dG9mZiA9IGhvdXJzWy0xXVs6MTBdIGlmIGhvdXJzIGVsc2UgIiIKICAgIHRlc3RfZnJvbSA9IChkYXRldGltZS5zdHJwdGltZShjdXRvZmYsICIlWS0lbS0lZCIpIC0gdGltZWRlbHRhKGRheXM9VEVTVF9EQVlTIC0gMSkpLnN0cmZ0aW1lKCIlWS0lbS0lZCIpIGlmIGhvdXJzIGVsc2UgIiIKICAgIGZvciB0IGluIGhvdXJzOgogICAgICAgIHByZXYgPSAoZGF0ZXRpbWUuc3RycHRpbWUodCwgIiVZLSVtLSVkVCVIOiVNIikgLSB0aW1lZGVsdGEoaG91cnM9MjQpKS5zdHJmdGltZSgiJVktJW0tJWRUJUg6JU0iKQogICAgICAgIGlmIHByZXYgbm90IGluIG9ic193YjoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBULCBSSCwgXyA9IGZjW3RdCiAgICAgICAgcF93YiA9IGZjX3diW3RdCiAgICAgICAgcmVzaWR1YWwgPSBvYnNfd2JbcHJldl0gLSBmY193YltwcmV2XSBpZiBwcmV2IGluIGZjX3diIGVsc2UgMC4wICAgIyBob3cgd3JvbmcgdGhlIHByb3ZpZGVyIHdhcyBhIGRheSBhZ28KICAgICAgICByb3dzLmFwcGVuZCh3ZXRidWxiX3Jvdyh0LCAwLCBwX3diLCBULCBSSCwgcmVjZW50X3Jlc2lkdWFsPXJlc2lkdWFsKSkKICAgICAgICB5LmFwcGVuZChvYnNfd2JbdF0pOyBwZXJzLmFwcGVuZChvYnNfd2JbcHJldl0pOyBwcm92LmFwcGVuZChwX3diKTsgdGVzdC5hcHBlbmQodFs6MTBdID49IHRlc3RfZnJvbSkKICAgIHJldHVybiByb3dzLCB5LCBwZXJzLCBwcm92LCB0ZXN0CgoKZGVmIGNhcmJvbl9kYXRhc2V0KHJlZ2lvbl9pZDogc3RyLCBsZWFkcz0oMSwgNiwgMTIsIDI0KSk6CiAgICBwYXRoID0gUk9PVCAvICJkYXRhIiAvICJoaXN0b3J5IiAvIGYie3JlZ2lvbl9pZH0uY3N2IgogICAgaWYgbm90IHBhdGguZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHdpdGggcGF0aC5vcGVuKG5ld2xpbmU9IiIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgcmVhbCA9IHtyWyJob3VyIl06IGZsb2F0KHJbImNpX2dfcGVyX2t3aCJdKSBmb3IgciBpbiBjc3YuRGljdFJlYWRlcihmKSBpZiByWyJjaV9zb3VyY2UiXSAhPSAibW9kZWxsZWQifQogICAgaG91cnMgPSBzb3J0ZWQocmVhbCkKICAgIGlmIGxlbihob3VycykgPCAyNCAqIDEwOgogICAgICAgIHJldHVybiBOb25lCiAgICBmbXQgPSAiJVktJW0tJWRUJUg6MDAiCiAgICB0ZXN0X2Zyb20gPSAoZGF0ZXRpbWUuc3RycHRpbWUoaG91cnNbLTFdLCBmbXQpIC0gdGltZWRlbHRhKGRheXM9VEVTVF9EQVlTKSkuc3RyZnRpbWUoZm10KQogICAgcm93cywgeSwgcGVycywgcHJvdiwgdGVzdCA9IFtdLCBbXSwgW10sIFtdLCBbXQogICAgZm9yIGlzc3VlIGluIGhvdXJzOgogICAgICAgIGZvciBsZWFkIGluIGxlYWRzOgogICAgICAgICAgICB0YXJnZXQgPSAoZGF0ZXRpbWUuc3RycHRpbWUoaXNzdWUsIGZtdCkgKyB0aW1lZGVsdGEoaG91cnM9bGVhZCkpLnN0cmZ0aW1lKGZtdCkKICAgICAgICAgICAgeWRheSA9IChkYXRldGltZS5zdHJwdGltZSh0YXJnZXQsIGZtdCkgLSB0aW1lZGVsdGEoaG91cnM9MjQpKS5zdHJmdGltZShmbXQpCiAgICAgICAgICAgIGlmIHRhcmdldCBub3QgaW4gcmVhbCBvciB5ZGF5IG5vdCBpbiByZWFsOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcm93cy5hcHBlbmQoY2FyYm9uX3Jvdyh0YXJnZXQsIGxlYWQsIHJlYWxbaXNzdWVdLCByZWFsW3lkYXldKSkKICAgICAgICAgICAgeS5hcHBlbmQocmVhbFt0YXJnZXRdKTsgcGVycy5hcHBlbmQocmVhbFtpc3N1ZV0pOyBwcm92LmFwcGVuZChyZWFsW2lzc3VlXSk7IHRlc3QuYXBwZW5kKHRhcmdldCA+PSB0ZXN0X2Zyb20pCiAgICByZXR1cm4gcm93cywgeSwgcGVycywgcHJvdiwgdGVzdAoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLWRheXMiLCB0eXBlPWludCwgZGVmYXVsdD02MCkKICAgIHAuYWRkX2FyZ3VtZW50KCItLXJlZ2lvbnMiLCBoZWxwPSJjb21tYS1zZXBhcmF0ZWQgKGRlZmF1bHQ6IGFsbCkiKQogICAgYXJncyA9IHAucGFyc2VfYXJncygpCiAgICBPVVQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgd2FudGVkID0gYXJncy5yZWdpb25zLnNwbGl0KCIsIikgaWYgYXJncy5yZWdpb25zIGVsc2UgbGlzdChyZWdpb25fY29uZmlnLmxvYWQoKSkKICAgIG1ldHJpY3MgPSBbXQogICAgZm9yIHJpZCBpbiB3YW50ZWQ6CiAgICAgICAgcmVnaW9uID0gcmVnaW9uX2NvbmZpZy5nZXQocmlkKQogICAgICAgIGZvciB0YXJnZXQsIGZlYXR1cmVzLCBkYXRhIGluICgoInRfd2IiLCBXRVRCVUxCX0ZFQVRVUkVTLCBsYW1iZGE6IHdldGJ1bGJfZGF0YXNldChyZWdpb24sIGFyZ3MuZGF5cykpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAoImNpIiwgQ0FSQk9OX0ZFQVRVUkVTLCBsYW1iZGE6IGNhcmJvbl9kYXRhc2V0KHJpZCkpKToKICAgICAgICAgICAgZCA9IGRhdGEoKQogICAgICAgICAgICBpZiBkIGlzIE5vbmU6CiAgICAgICAgICAgICAgICBwcmludChmIntyaWQ6MTZ9IHt0YXJnZXQ6NX0gc2tpcHBlZDogbm90IGVub3VnaCByZWFsIGhpc3RvcnkgeWV0IikKICAgICAgICAgICAgICAgIG1ldHJpY3MuYXBwZW5kKHsicmVnaW9uIjogcmlkLCAidGFyZ2V0IjogdGFyZ2V0LCAic2tpcHBlZCI6ICJub3QgZW5vdWdoIHJlYWwgaGlzdG9yeSJ9KQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgcm93cywgeSwgcGVyc2lzdGVuY2UsIHByb3ZpZGVyLCB0ZXN0ID0gZAogICAgICAgICAgICAgICAgdmVyZGljdCwgbW9kZWwgPSBmaXRfYW5kX2p1ZGdlKHRhcmdldCwgcmlkLCByb3dzLCB5LCBmZWF0dXJlcywgcGVyc2lzdGVuY2UsIHByb3ZpZGVyLCB0ZXN0KQogICAgICAgICAgICBleGNlcHQgVmFsdWVFcnJvciBhcyBlOgogICAgICAgICAgICAgICAgcHJpbnQoZiJ7cmlkOjE2fSB7dGFyZ2V0OjV9IHNraXBwZWQ6IHtlfSIpCiAgICAgICAgICAgICAgICBtZXRyaWNzLmFwcGVuZCh7InJlZ2lvbiI6IHJpZCwgInRhcmdldCI6IHRhcmdldCwgInNraXBwZWQiOiBzdHIoZSl9KQogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgdiA9IHZlcmRpY3QuYXNfZGljdCgpCiAgICAgICAgICAgIG1ldHJpY3MuYXBwZW5kKHYpCiAgICAgICAgICAgIHByaW50KGYie3JpZDoxNn0ge3RhcmdldDo1fSBNQUUgbW9kZWwge3ZbJ21hZV9tb2RlbCddOi4zZn0gfCBwZXJzaXN0ZW5jZSB7dlsnbWFlX3BlcnNpc3RlbmNlJ106LjNmfSB8ICIKICAgICAgICAgICAgICAgICAgZiJwcm92aWRlciB7dlsnbWFlX3Byb3ZpZGVyJ106LjNmfSAtPiB7J1VTRScgaWYgdlsndXNlX21vZGVsJ10gZWxzZSAnbm90IHVzZWQnfSIpCiAgICAgICAgICAgIHBhdGggPSBPVVQgLyBmIntyaWR9LXt0YXJnZXR9Lmpzb24iCiAgICAgICAgICAgIGlmIHZlcmRpY3QudXNlX21vZGVsOgogICAgICAgICAgICAgICAgcGF0aC53cml0ZV90ZXh0KGpzb24uZHVtcHMoeyoqbW9kZWwsICJ2ZXJkaWN0Ijogdn0pICsgIlxuIikKICAgICAgICAgICAgZWxpZiBwYXRoLmV4aXN0cygpOgogICAgICAgICAgICAgICAgcGF0aC51bmxpbmsoKQogICAgKE9VVCAvICJtZXRyaWNzLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoewogICAgICAgICJ0cmFpbmVkX2F0IjogZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0Yykuc3RyZnRpbWUoIiVZLSVtLSVkVCVIOiVNWiIpLAogICAgICAgICJ0ZXN0X2RheXMiOiBURVNUX0RBWVMsICJyZXN1bHRzIjogbWV0cmljc30sIGluZGVudD0xKSArICJcbiIpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBzeXMuZXhpdChtYWluKCkpCg==",
"sources/__init__.py": "IiIiRXh0ZXJuYWwgZGF0YTogd2VhdGhlciAoT3Blbi1NZXRlbykgYW5kIGdyaWQgY2FyYm9uL21peCAoRWxlY3RyaWNpdHkgTWFwcywgb3IgYSBtb2RlbGxlZCBmYWxsYmFjaykuIiIiCg==",
"sources/openmeteo.py": "IiIiV2VhdGhlciBmb3Igb25lIGhvdXIgYXQgb25lIHBvaW50IGZyb20gT3Blbi1NZXRlbyAoc3RkbGliIG9ubHkpLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IHRpbWUKaW1wb3J0IHVybGxpYi5lcnJvcgppbXBvcnQgdXJsbGliLnBhcnNlCmltcG9ydCB1cmxsaWIucmVxdWVzdApmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZSwgdGltZWRlbHRhLCB0aW1lem9uZQoKRk9SRUNBU1RfVVJMID0gImh0dHBzOi8vYXBpLm9wZW4tbWV0ZW8uY29tL3YxL2ZvcmVjYXN0IgpBUkNISVZFX1VSTCA9ICJodHRwczovL2FyY2hpdmUtYXBpLm9wZW4tbWV0ZW8uY29tL3YxL2FyY2hpdmUiClZBUlMgPSAoInRlbXBlcmF0dXJlXzJtIiwgInJlbGF0aXZlX2h1bWlkaXR5XzJtIiwgInN1cmZhY2VfcHJlc3N1cmUiKQoKClJFVFJJRVMgPSAzICAgICAgICAgICMgZXh0cmEgYXR0ZW1wdHMgZm9yIHRyYW5zaWVudCBmYWlsdXJlcyAoZHJvcHBlZCBjb25uZWN0aW9uLCA1eHgsIHBlci1taW51dGUgNDI5KQpCQUNLT0ZGX1MgPSAoMi4wLCA1LjAsIDEyLjApCgoKZGVmIF9nZXQodXJsOiBzdHIsIHBhcmFtczogZGljdCwgdGltZW91dDogZmxvYXQgPSAxNS4wKSAtPiBkaWN0OgogICAgcmVxID0gdXJsbGliLnJlcXVlc3QuUmVxdWVzdChmInt1cmx9P3t1cmxsaWIucGFyc2UudXJsZW5jb2RlKHBhcmFtcyl9IiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaGVhZGVycz17IlVzZXItQWdlbnQiOiAicHJhdmFhaC8wLjEifSkKICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKFJFVFJJRVMgKyAxKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3BlbihyZXEsIHRpbWVvdXQ9dGltZW91dCkgYXMgcmVzcDoKICAgICAgICAgICAgICAgIHJldHVybiBqc29uLmxvYWQocmVzcCkKICAgICAgICBleGNlcHQgdXJsbGliLmVycm9yLkhUVFBFcnJvciBhcyBlOgogICAgICAgICAgICBib2R5ID0gZS5yZWFkKCkuZGVjb2RlKCJ1dGYtOCIsICJyZXBsYWNlIilbOjMwMF0KICAgICAgICAgICAgdHJhbnNpZW50ID0gZS5jb2RlIGluICg1MDAsIDUwMiwgNTAzLCA1MDQpIG9yIChlLmNvZGUgPT0gNDI5IGFuZCAiZGFpbHkiIG5vdCBpbiBib2R5Lmxvd2VyKCkpCiAgICAgICAgICAgIGlmIG5vdCB0cmFuc2llbnQgb3IgYXR0ZW1wdCA9PSBSRVRSSUVTOgogICAgICAgICAgICAgICAgcmFpc2UKICAgICAgICBleGNlcHQgKHVybGxpYi5lcnJvci5VUkxFcnJvciwgVGltZW91dEVycm9yLCBDb25uZWN0aW9uRXJyb3IpOgogICAgICAgICAgICBpZiBhdHRlbXB0ID09IFJFVFJJRVM6CiAgICAgICAgICAgICAgICByYWlzZQogICAgICAgIHRpbWUuc2xlZXAoQkFDS09GRl9TW2F0dGVtcHRdKQoKCmRlZiBwYXJzZV9ob3VyKHRleHQ6IHN0cikgLT4gZGF0ZXRpbWU6CiAgICAiIiJJU08gaG91ciwgVVRDLiBBIHRyYWlsaW5nIFogb3Igb2Zmc2V0IGlzIGhvbm91cmVkOyBuYWl2ZSBtZWFucyBVVEMuIiIiCiAgICBkdCA9IGRhdGV0aW1lLmZyb21pc29mb3JtYXQodGV4dC5yZXBsYWNlKCJaIiwgIiswMDowMCIpKQogICAgaWYgZHQudHppbmZvIGlzIG5vdCBOb25lOgogICAgICAgIGR0ID0gZHQuYXN0aW1lem9uZSh0aW1lem9uZS51dGMpLnJlcGxhY2UodHppbmZvPU5vbmUpCiAgICByZXR1cm4gZHQucmVwbGFjZShtaW51dGU9MCwgc2Vjb25kPTAsIG1pY3Jvc2Vjb25kPTApCgoKZGVmIGhvdXJfd2VhdGhlcihsYXQ6IGZsb2F0LCBsb246IGZsb2F0LCBob3VyX3V0YzogZGF0ZXRpbWUpIC0+IHR1cGxlW2Zsb2F0LCBmbG9hdCwgZmxvYXRdOgogICAgIiIiKHRlbXBlcmF0dXJlIMKwQywgUkggJSwgc3VyZmFjZSBwcmVzc3VyZSBoUGEpIGZvciBvbmUgVVRDIGhvdXIgKGZvcmVjYXN0IG9yIHJlY2VudCBwYXN0KS4iIiIKICAgIHN0YW1wID0gaG91cl91dGMuc3RyZnRpbWUoIiVZLSVtLSVkVCVIOjAwIikKICAgIHJhdyA9IF9nZXQoRk9SRUNBU1RfVVJMLCB7CiAgICAgICAgImxhdGl0dWRlIjogbGF0LCAibG9uZ2l0dWRlIjogbG9uLCAiaG91cmx5IjogIiwiLmpvaW4oVkFSUyksCiAgICAgICAgInRpbWV6b25lIjogIlVUQyIsICJzdGFydF9ob3VyIjogc3RhbXAsICJlbmRfaG91ciI6IHN0YW1wLAogICAgfSkKICAgIGggPSByYXdbImhvdXJseSJdCiAgICByZXR1cm4gaFsidGVtcGVyYXR1cmVfMm0iXVswXSwgaFsicmVsYXRpdmVfaHVtaWRpdHlfMm0iXVswXSwgaFsic3VyZmFjZV9wcmVzc3VyZSJdWzBdCgoKZGVmIGhpc3RvcnkobGF0OiBmbG9hdCwgbG9uOiBmbG9hdCwgc3RhcnQ6IGRhdGV0aW1lLCBlbmQ6IGRhdGV0aW1lKSAtPiBsaXN0W3R1cGxlW3N0ciwgZmxvYXQsIGZsb2F0LCBmbG9hdF1dOgogICAgIiIiSG91cmx5ICh0aW1lLCBULCBSSCwgUCkgZnJvbSB0aGUgYXJjaGl2ZSBBUEkgYmV0d2VlbiB0d28gVVRDIGRhdGVzIGluY2x1c2l2ZS4iIiIKICAgIHJhdyA9IF9nZXQoQVJDSElWRV9VUkwsIHsKICAgICAgICAibGF0aXR1ZGUiOiBsYXQsICJsb25naXR1ZGUiOiBsb24sICJob3VybHkiOiAiLCIuam9pbihWQVJTKSwgInRpbWV6b25lIjogIlVUQyIsCiAgICAgICAgInN0YXJ0X2RhdGUiOiBzdGFydC5zdHJmdGltZSgiJVktJW0tJWQiKSwgImVuZF9kYXRlIjogZW5kLnN0cmZ0aW1lKCIlWS0lbS0lZCIpLAogICAgfSwgdGltZW91dD02MC4wKQogICAgaCA9IHJhd1siaG91cmx5Il0KICAgIHJldHVybiBbCiAgICAgICAgKHQsIHRlbXAsIHJoLCBwKQogICAgICAgIGZvciB0LCB0ZW1wLCByaCwgcCBpbiB6aXAoaFsidGltZSJdLCBoWyJ0ZW1wZXJhdHVyZV8ybSJdLCBoWyJyZWxhdGl2ZV9odW1pZGl0eV8ybSJdLCBoWyJzdXJmYWNlX3ByZXNzdXJlIl0pCiAgICAgICAgaWYgTm9uZSBub3QgaW4gKHRlbXAsIHJoLCBwKQogICAgXQoKCmRlZiBsYXN0X2Z1bGxfeWVhcih0b2RheTogZGF0ZXRpbWUgfCBOb25lID0gTm9uZSkgLT4gdHVwbGVbZGF0ZXRpbWUsIGRhdGV0aW1lXToKICAgIHRvZGF5ID0gdG9kYXkgb3IgZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0YykucmVwbGFjZSh0emluZm89Tm9uZSkKICAgIGVuZCA9IHRvZGF5IC0gdGltZWRlbHRhKGRheXM9NykgICMgYXJjaGl2ZSBsYWdzIGFib3V0IDUgZGF5cwogICAgcmV0dXJuIGVuZCAtIHRpbWVkZWx0YShkYXlzPTM2NCksIGVuZAoKCmRlZiBmb3JlY2FzdChsYXQ6IGZsb2F0LCBsb246IGZsb2F0LCBob3VyczogaW50ID0gNDgpIC0+IGxpc3RbdHVwbGVbc3RyLCBmbG9hdCwgZmxvYXQsIGZsb2F0XV06CiAgICAiIiJOZXh0IGBob3Vyc2AgaG91cmx5IChVVEMgdGltZSwgVCwgUkgsIFApLCBzdGFydGluZyBhdCB0aGUgY3VycmVudCBVVEMgaG91ci4iIiIKICAgIHJhdyA9IF9nZXQoRk9SRUNBU1RfVVJMLCB7CiAgICAgICAgImxhdGl0dWRlIjogbGF0LCAibG9uZ2l0dWRlIjogbG9uLCAiaG91cmx5IjogIiwiLmpvaW4oVkFSUyksICJ0aW1lem9uZSI6ICJVVEMiLAogICAgICAgICJmb3JlY2FzdF9kYXlzIjogbWF4KDIsIChob3VycyArIDQ3KSAvLyAyNCksCiAgICB9KQogICAgbm93ID0gZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0Yykuc3RyZnRpbWUoIiVZLSVtLSVkVCVIOjAwIikKICAgIGggPSByYXdbImhvdXJseSJdCiAgICByb3dzID0gWwogICAgICAgICh0LCB0ZW1wLCByaCwgcCkKICAgICAgICBmb3IgdCwgdGVtcCwgcmgsIHAgaW4gemlwKGhbInRpbWUiXSwgaFsidGVtcGVyYXR1cmVfMm0iXSwgaFsicmVsYXRpdmVfaHVtaWRpdHlfMm0iXSwgaFsic3VyZmFjZV9wcmVzc3VyZSJdKQogICAgICAgIGlmIHQgPj0gbm93IGFuZCBOb25lIG5vdCBpbiAodGVtcCwgcmgsIHApCiAgICBdCiAgICByZXR1cm4gcm93c1s6aG91cnNdCgoKZGVmIHJlY2VudChsYXQ6IGZsb2F0LCBsb246IGZsb2F0LCBkYXlzOiBpbnQpIC0+IGxpc3RbdHVwbGVbc3RyLCBmbG9hdCwgZmxvYXQsIGZsb2F0XV06CiAgICAiIiJIb3VybHkgKFVUQyB0aW1lLCBULCBSSCwgUCkgZm9yIHRoZSBsYXN0IGBkYXlzYCBkYXlzIHVwIHRvIHRoZSBjdXJyZW50IGhvdXIuCgogICAgVXNlcyB0aGUgZm9yZWNhc3QgZW5kcG9pbnQncyBwYXN0X2RheXMgKG1heCA5MiksIHdoaWNoIGhhcyBubyBhcmNoaXZlIGxhZywgc28gaXQKICAgIG92ZXJsYXBzIHdpdGggdGhlIGxhc3QtMjQgaCBjYXJib24gaGlzdG9yeSB0aGUgRWxlY3RyaWNpdHkgTWFwcyBmcmVlIHRpZXIgZ2l2ZXMuCiAgICAiIiIKICAgIGlmIG5vdCAxIDw9IGRheXMgPD0gOTI6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZGF5cyBtdXN0IGJlIDEtOTIgKE9wZW4tTWV0ZW8gcGFzdF9kYXlzIGxpbWl0KTsgdXNlIGhpc3RvcnkoKSBmb3Igb2xkZXIgZGF0YSIpCiAgICByYXcgPSBfZ2V0KEZPUkVDQVNUX1VSTCwgewogICAgICAgICJsYXRpdHVkZSI6IGxhdCwgImxvbmdpdHVkZSI6IGxvbiwgImhvdXJseSI6ICIsIi5qb2luKFZBUlMpLCAidGltZXpvbmUiOiAiVVRDIiwKICAgICAgICAicGFzdF9kYXlzIjogZGF5cywgImZvcmVjYXN0X2RheXMiOiAxLAogICAgfSwgdGltZW91dD02MC4wKQogICAgbm93ID0gZGF0ZXRpbWUubm93KHRpbWV6b25lLnV0Yykuc3RyZnRpbWUoIiVZLSVtLSVkVCVIOjAwIikKICAgIGggPSByYXdbImhvdXJseSJdCiAgICByZXR1cm4gWwogICAgICAgICh0LCB0ZW1wLCByaCwgcCkKICAgICAgICBmb3IgdCwgdGVtcCwgcmgsIHAgaW4gemlwKGhbInRpbWUiXSwgaFsidGVtcGVyYXR1cmVfMm0iXSwgaFsicmVsYXRpdmVfaHVtaWRpdHlfMm0iXSwgaFsic3VyZmFjZV9wcmVzc3VyZSJdKQogICAgICAgIGlmIHQgPD0gbm93IGFuZCBOb25lIG5vdCBpbiAodGVtcCwgcmgsIHApCiAgICBdCg==",
"model/__init__.py": "IiIiUHJhdmFhaCBjb3N0IG1vZGVsOiBsaXRyZXMgb2Ygd2F0ZXIgYW5kIGtnIG9mIENPMiBwZXIgam9iLCBwZXIgcmVnaW9uLCBwZXIgaG91ci4iIiIK",
"model/coefficients.py": "IiIiTG9hZCBtb2RlbC9jb2VmZmljaWVudHMueWFtbC4gRXZlcnkgbnVtYmVyIHRoZSBtb2RlbCB1c2VzIGNvbWVzIHRocm91Z2ggaGVyZS4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZnVuY3Rvb2xzIGltcG9ydCBscnVfY2FjaGUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgeWFtbAoKUEFUSCA9IFBhdGgoX19maWxlX18pLndpdGhfbmFtZSgiY29lZmZpY2llbnRzLnlhbWwiKQoKCkBscnVfY2FjaGUobWF4c2l6ZT0xKQpkZWYgbG9hZCgpIC0+IGRpY3Q6CiAgICB3aXRoIFBBVEgub3BlbihlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHJldHVybiB5YW1sLnNhZmVfbG9hZChmKQoKCmRlZiB2YWx1ZSgqcGF0aDogc3RyKSAtPiBmbG9hdDoKICAgICIiInZhbHVlKCJzaXRlX2Nvb2xpbmciLCAidG93ZXIiLCAiY3ljbGVzX29mX2NvbmNlbnRyYXRpb24iKSAtPiA1IiIiCiAgICBub2RlID0gbG9hZCgpCiAgICBmb3Iga2V5IGluIHBhdGg6CiAgICAgICAgbm9kZSA9IG5vZGVba2V5XQogICAgcmV0dXJuIG5vZGVbInZhbHVlIl0gaWYgaXNpbnN0YW5jZShub2RlLCBkaWN0KSBhbmQgInZhbHVlIiBpbiBub2RlIGVsc2Ugbm9kZQoKCmRlZiB1bmNlcnRhaW50eSgqcGF0aDogc3RyKSAtPiBmbG9hdDoKICAgIG5vZGUgPSBsb2FkKCkKICAgIGZvciBrZXkgaW4gcGF0aDoKICAgICAgICBub2RlID0gbm9kZVtrZXldCiAgICByZXR1cm4gZmxvYXQobm9kZS5nZXQoInVuY2VydGFpbnR5IiwgMC4wKSkK",
"model/coefficients.yaml": "IyBFdmVyeSBudW1iZXIgdGhlIGNvc3QgbW9kZWwgdXNlcy4gTm90aGluZyBudW1lcmljIGxpdmVzIGluIHRoZSBQeXRob24gY29kZS4KIwojIEVhY2ggY29lZmZpY2llbnQgaGFzOgojICAgdmFsdWUgICAgICAgIHRoZSBudW1iZXIKIyAgIHVuaXQKIyAgIHVuY2VydGFpbnR5ICByZWxhdGl2ZSBoYWxmLXdpZHRoIG9mIGEgcGxhdXNpYmxlIHJhbmdlICgwLjIgPSDCsTIwJSksIHVzZWQgZm9yIHJlY2VpcHQgYmFuZHMKIyAgIHNvdXJjZSAgICAgICB7dGl0bGUsIHVybH0KIyAgIG5vdGUgICAgICAgICB3aHkgdGhpcyB2YWx1ZSwgYW5kIGl0cyBsaW1pdHMKIyAgIGNoZWNrZWQgICAgICB0cnVlIG9uY2UgYSB0ZWFtIG1lbWJlciBoYXMgb3BlbmVkIHRoZSBzb3VyY2UgYW5kIGNvbmZpcm1lZCB0aGUgdmFsdWUuCiMgICAgICAgICAgICAgICAgRXZlcnl0aGluZyBzdGFydHMgZmFsc2U6IHRoZSBidWlsZCBlbnZpcm9ubWVudCBjb3VsZCBub3Qgb3BlbiBtb3N0IHNvdXJjZXMsCiMgICAgICAgICAgICAgICAgc28gdmFsdWVzIGNvbWUgZnJvbSB0aGUgY2l0ZWQgZG9jdW1lbnRzIGFzIHN1bW1hcmlzZWQgYnkgc2VhcmNoIHJlc3VsdHMsCiMgICAgICAgICAgICAgICAgb3IgZnJvbSBzdGFuZGFyZCBwaHlzaWNzIHJlZmVyZW5jZXMuCiMKIyBgcHl0aG9uIHNjcmlwdHMvdmFsaWRhdGVfZGF0YS5weWAgZW5mb3JjZXMgdGhpcyBzaGFwZS4KCnBoeXNpY3M6CiAgbGF0ZW50X2hlYXRfdmFwb3Jpc2F0aW9uOgogICAgdmFsdWU6IDIuNDMKICAgIHVuaXQ6IE1KL2tnCiAgICB1bmNlcnRhaW50eTogMC4wMQogICAgc291cmNlOgogICAgICB0aXRsZTogSUFQV1MtSUY5NyBzdGVhbSB0YWJsZXMgKGxhdGVudCBoZWF0IG9mIHdhdGVyIGF0IH4zMCDCsEMpCiAgICAgIHVybDogaHR0cDovL3d3dy5pYXB3cy5vcmcvcmVsZ3VpZGUvSUY5Ny1SZXYuaHRtbAogICAgbm90ZTogRW5lcmd5IHRvIGV2YXBvcmF0ZSAxIGtnIG9mIHdhdGVyIGF0IGNvb2xpbmctdG93ZXIgdGVtcGVyYXR1cmVzLiAxIGtXaCBvZiBoZWF0IGV2YXBvcmF0ZXMgMy42LzIuNDMgPSAxLjQ4IGtnLgogICAgY2hlY2tlZDogZmFsc2UKICBhaXJfc3BlY2lmaWNfaGVhdDoKICAgIHZhbHVlOiAxLjAwNgogICAgdW5pdDoga0ovKGtnwrdLKQogICAgdW5jZXJ0YWludHk6IDAuMDEKICAgIHNvdXJjZToKICAgICAgdGl0bGU6IEVuZ2luZWVyaW5nIFRvb2xCb3gsIGFpciBzcGVjaWZpYyBoZWF0IGF0IGNvbnN0YW50IHByZXNzdXJlCiAgICAgIHVybDogaHR0cHM6Ly93d3cuZW5naW5lZXJpbmd0b29sYm94LmNvbS9haXItc3BlY2lmaWMtaGVhdC1jYXBhY2l0eS1kXzcwNS5odG1sCiAgICBub3RlOiBEcnkgYWlyIG5lYXIgcm9vbSB0ZW1wZXJhdHVyZS4KICAgIGNoZWNrZWQ6IGZhbHNlCiAgcHN5Y2hyb21ldGVyX2NvZWZmaWNpZW50OgogICAgdmFsdWU6IDYuNTNlLTQKICAgIHVuaXQ6IDEvSwogICAgdW5jZXJ0YWludHk6IDAuMDIKICAgIHNvdXJjZToKICAgICAgdGl0bGU6IFdNTy1Oby4gOCwgR3VpZGUgdG8gSW5zdHJ1bWVudHMgYW5kIE1ldGhvZHMgb2YgT2JzZXJ2YXRpb24sIEFubmV4IDQuQiAoQXNzbWFubiBwc3ljaHJvbWV0ZXIpCiAgICAgIHVybDogaHR0cHM6Ly9saWJyYXJ5Lndtby5pbnQvaWR1cmwvNC80MTY1MAogICAgbm90ZTogIkEgaW4gZSA9IGVfdyhUdykgLSBBwrcoMSArIDAuMDAwOTQ0wrdUdynCt1DCtyhUIC0gVHcpLCBmb3IgYSB2ZW50aWxhdGVkIHBzeWNocm9tZXRlci4iCiAgICBjaGVja2VkOiBmYWxzZQoKc2l0ZV9jb29saW5nOgogICMgT24tc2l0ZSB3YXRlciBwZXIga1doIG9mIElUIGVuZXJneSAodGhlIFdVRSBkZWZpbml0aW9uKSwgYnkgY29vbGluZyBkZXNpZ24uCiAgIyBUaGVzZSBjdXJ2ZXMgc2V0IHRoZSAqc2hhcGUqIG92ZXIgdGhlIGRheTsgZWFjaCByZWdpb24ncyBsZXZlbCBpcyBjYWxpYnJhdGVkCiAgIyB0byBpdHMgZGlzY2xvc2VkIGFubnVhbCBXVUUgKHNlZSByZWdpb25hbF93dWUgYmVsb3cgYW5kIG1vZGVsL3dhdGVyLnB5KS4KICB0b3dlcjoKICAgIGhlYXRfcmVqZWN0ZWRfcGVyX2l0X2t3aDoKICAgICAgdmFsdWU6IDEuMgogICAgICB1bml0OiBrV2ggaGVhdCAvIGtXaCBJVAogICAgICB1bmNlcnRhaW50eTogMC4xCiAgICAgIHNvdXJjZToKICAgICAgICB0aXRsZTogIkFTSFJBRSBUQyA5Ljk6IGNoaWxsZXIgQ09QIH41IGFkZHMgfjAuMiBrV2ggb2YgY29tcHJlc3NvciBoZWF0IHBlciBrV2ggb2YgSVQgaGVhdCIKICAgICAgICB1cmw6IGh0dHBzOi8vd3d3LmFzaHJhZS5vcmcvdGVjaG5pY2FsLXJlc291cmNlcy9ib29rc3RvcmUvZGF0YWNvbS1zZXJpZXMKICAgICAgbm90ZTogVG93ZXIgcmVqZWN0cyBJVCBoZWF0IHBsdXMgY2hpbGxlciBjb21wcmVzc29yIHdvcmsuCiAgICAgIGNoZWNrZWQ6IGZhbHNlCiAgICBjeWNsZXNfb2ZfY29uY2VudHJhdGlvbjoKICAgICAgdmFsdWU6IDUKICAgICAgdW5pdDogY3ljbGVzCiAgICAgIHVuY2VydGFpbnR5OiAwLjQKICAgICAgc291cmNlOgogICAgICAgIHRpdGxlOiAiTGksIFlhbmcsIElzbGFtLCBSZW4gKDIwMjMpLCBNYWtpbmcgQUkgTGVzcyAnVGhpcnN0eScsIGFyWGl2OjIzMDQuMDMyNzEgKHdhdGVyIHJlY3ljbGVkIDMtMTAgY3ljbGVzKSIKICAgICAgICB1cmw6IGh0dHBzOi8vYXJ4aXYub3JnL2Ficy8yMzA0LjAzMjcxCiAgICAgIG5vdGU6ICJNYWtlLXVwIHdhdGVyID0gZXZhcG9yYXRpb24gw5cgQy8oQy0xKS4gNSBjeWNsZXMgYWRkcyAyNSUgYmxvd2Rvd24uIFJhbmdlIDMtMTAgZHJpdmVzIG1vc3Qgb2YgdGhlIHVuY2VydGFpbnR5LiIKICAgICAgY2hlY2tlZDogZmFsc2UKICAgIGZyZWVfY29vbGluZ19iZWxvd193ZXRidWxiX2M6CiAgICAgIHZhbHVlOiAxMC4wCiAgICAgIHVuaXQ6IMKwQyB3ZXQtYnVsYgogICAgICB1bmNlcnRhaW50eTogMC4zCiAgICAgIHNvdXJjZToKICAgICAgICB0aXRsZTogIkxlaSAmIE1hc2FuZXQgKDIwMjIpLCBDbGltYXRlLSBhbmQgdGVjaG5vbG9neS1zcGVjaWZpYyBQVUUgYW5kIFdVRSBlc3RpbWF0aW9ucyBmb3IgVS5TLiBkYXRhIGNlbnRlcnMsIFJlc291cmNlcywgQ29uc2VydmF0aW9uICYgUmVjeWNsaW5nIDE4MjoxMDYzMjMiCiAgICAgICAgdXJsOiBodHRwczovL2RvaS5vcmcvMTAuMTAxNi9qLnJlc2NvbnJlYy4yMDIyLjEwNjMyMwogICAgICBub3RlOiBXYXRlci1zaWRlIGVjb25vbWlzZXIgY292ZXJzIHRoZSBsb2FkIHdpdGggbm8gZXZhcG9yYXRpb24gYmVsb3cgdGhpcyB3ZXQtYnVsYiAoY2hpbGxlZC13YXRlciBzdXBwbHkgfjE4IMKwQyBtaW51cyB0b3dlciBhcHByb2FjaCkuCiAgICAgIGNoZWNrZWQ6IGZhbHNlCiAgICBmdWxsX2V2YXBvcmF0aXZlX2Fib3ZlX3dldGJ1bGJfYzoKICAgICAgdmFsdWU6IDE4LjAKICAgICAgdW5pdDogwrBDIHdldC1idWxiCiAgICAgIHVuY2VydGFpbnR5OiAwLjIKICAgICAgc291cmNlOgogICAgICAgIHRpdGxlOiAiTGVpICYgTWFzYW5ldCAoMjAyMiksIGFzIGFib3ZlIgogICAgICAgIHVybDogaHR0cHM6Ly9kb2kub3JnLzEwLjEwMTYvai5yZXNjb25yZWMuMjAyMi4xMDYzMjMKICAgICAgbm90ZTogQWJvdmUgdGhpcywgYWxsIGhlYXQgbGVhdmVzIGJ5IGV2YXBvcmF0aW9uLiBMaW5lYXIgcmFtcCBpbiBiZXR3ZWVuLgogICAgICBjaGVja2VkOiBmYWxzZQogIGh5YnJpZDoKICAgIHdldGJ1bGJfc2hpZnRfYzoKICAgICAgdmFsdWU6IDUuMAogICAgICB1bml0OiDCsEMKICAgICAgdW5jZXJ0YWludHk6IDAuNQogICAgICBzb3VyY2U6CiAgICAgICAgdGl0bGU6ICJMZWkgJiBNYXNhbmV0ICgyMDIyKSwgaHlicmlkIChkcnkgKyBldmFwb3JhdGl2ZSkgc3lzdGVtcyIKICAgICAgICB1cmw6IGh0dHBzOi8vZG9pLm9yZy8xMC4xMDE2L2oucmVzY29ucmVjLjIwMjIuMTA2MzIzCiAgICAgIG5vdGU6IEh5YnJpZCBjb29sZXJzIHJ1biBkcnkgZm9yIGxvbmdlcjsgbW9kZWxsZWQgYXMgdGhlIHRvd2VyIGN1cnZlIHNoaWZ0ZWQgdGhpcyBtdWNoIHdhcm1lci4gQ29hcnNlLgogICAgICBjaGVja2VkOiBmYWxzZQogIGFkaWFiYXRpYzoKICAgIHN1cHBseV9haXJfc2V0cG9pbnRfYzoKICAgICAgdmFsdWU6IDI3LjAKICAgICAgdW5pdDogwrBDIGRyeS1idWxiCiAgICAgIHVuY2VydGFpbnR5OiAwLjA1CiAgICAgIHNvdXJjZToKICAgICAgICB0aXRsZTogQVNIUkFFIFRoZXJtYWwgR3VpZGVsaW5lcyBmb3IgRGF0YSBQcm9jZXNzaW5nIEVudmlyb25tZW50cyAocmVjb21tZW5kZWQgaW5sZXQgMTgtMjcgwrBDKQogICAgICAgIHVybDogaHR0cHM6Ly93d3cuYXNocmFlLm9yZy90ZWNobmljYWwtcmVzb3VyY2VzL2Jvb2tzdG9yZS9kYXRhY29tLXNlcmllcwogICAgICBub3RlOiBEaXJlY3QgZXZhcG9yYXRpdmUgKGFkaWFiYXRpYykgYWlyIGNvb2xpbmcgb25seSBldmFwb3JhdGVzIHdhdGVyIHdoZW4gb3V0c2lkZSBhaXIgaXMgYWJvdmUgdGhpcy4KICAgICAgY2hlY2tlZDogZmFsc2UKICAgIHNlcnZlcl9haXJfZGVsdGFfdF9rOgogICAgICB2YWx1ZTogMTIuMAogICAgICB1bml0OiBLCiAgICAgIHVuY2VydGFpbnR5OiAwLjI1CiAgICAgIHNvdXJjZToKICAgICAgICB0aXRsZTogQVNIUkFFIFRDIDkuOSB0eXBpY2FsIHNlcnZlciBpbmxldC10by1vdXRsZXQgdGVtcGVyYXR1cmUgcmlzZQogICAgICAgIHVybDogaHR0cHM6Ly93d3cuYXNocmFlLm9yZy90ZWNobmljYWwtcmVzb3VyY2VzL2Jvb2tzdG9yZS9kYXRhY29tLXNlcmllcwogICAgICBub3RlOiBTZXRzIGFpcmZsb3cgcGVyIGtXaCAo4omIMjk4IGtnIGFpci9rV2ggYXQgMTIgSyksIGhlbmNlIHdhdGVyIHRvIGNvb2wgdGhhdCBhaXIuCiAgICAgIGNoZWNrZWQ6IGZhbHNlCiAgICBibGVlZF9mYWN0b3I6CiAgICAgIHZhbHVlOiAxLjE1CiAgICAgIHVuaXQ6IG11bHRpcGxpZXIKICAgICAgdW5jZXJ0YWludHk6IDAuMQogICAgICBzb3VyY2U6CiAgICAgICAgdGl0bGU6ICJMaSBldCBhbC4gKDIwMjMpLCBhclhpdjoyMzA0LjAzMjcxIChibG93ZG93biAyMC00MCUgb2YgY29vbGluZyB3YXRlcikiCiAgICAgICAgdXJsOiBodHRwczovL2FyeGl2Lm9yZy9hYnMvMjMwNC4wMzI3MQogICAgICBub3RlOiBFeHRyYSB3YXRlciBkcmFpbmVkIGZyb20gZXZhcG9yYXRpdmUgbWVkaWEgdG8gY29udHJvbCBtaW5lcmFscy4KICAgICAgY2hlY2tlZDogZmFsc2UKICBhaXI6CiAgICBub3RlOiBBaXItY29vbGVkIChkcnkgY29vbGVycywgYWlyLWNvb2xlZCBjaGlsbGVycykgdXNlcyBubyBvbi1zaXRlIHdhdGVyLgoKZ3JpZF93YXRlcl9mYWN0b3JzOgogICMgV2F0ZXIgY29uc3VtZWQgKG5vdCB3aXRoZHJhd24pIHBlciBrV2ggZ2VuZXJhdGVkLCBieSBzb3VyY2UsIG1hdGNoaW5nCiAgIyBFbGVjdHJpY2l0eSBNYXBzIHBvd2VyLWJyZWFrZG93biBjYXRlZ29yaWVzLiBVUyBtZWRpYW5zLCByZWNpcmN1bGF0aW5nCiAgIyAoY29vbGluZyB0b3dlcikgd2hlcmUgcmVsZXZhbnQ7IGFwcGxpZWQgZ2xvYmFsbHksIHdoaWNoIGlzIGEgc3RhdGVkIGxpbWl0YXRpb24uCiAgc291cmNlOgogICAgdGl0bGU6ICJNYWNrbmljaywgTmV3bWFyaywgSGVhdGgsIEhhbGxldHQgKDIwMTIpLCBPcGVyYXRpb25hbCB3YXRlciBjb25zdW1wdGlvbiBhbmQgd2l0aGRyYXdhbCBmYWN0b3JzIGZvciBlbGVjdHJpY2l0eSBnZW5lcmF0aW5nIHRlY2hub2xvZ2llcywgRW52aXJvbi4gUmVzLiBMZXR0LiA3OjA0NTgwMiAoTlJFTCkiCiAgICB1cmw6IGh0dHBzOi8vb3BlbmVpLm9yZy9kYXRhc2V0cy9maWxlcy85NjkvcHViLzE3NDgtOTMyNl83XzRfMDQ1ODAyLnBkZgogIHVuaXQ6IEwva1doCiAgdW5jZXJ0YWludHk6IDAuNQogIG5vdGU6ICJDb252ZXJ0ZWQgZnJvbSBnYWwvTVdoICjDlyAwLjAwMzc4NSkuIE1lZGlhbnMgdmFyeSB3aXRoIHBsYW50IHR5cGU6IG9uZSBzZWFyY2ggc3VtbWFyeSBnYXZlIGNvYWwgNDc5IGFuZCBnYXMgMjA1IGdhbC9NV2ggd2hlcmUgd2UgdXNlIHRoZSBnZW5lcmljLXRvd2VyIG1lZGlhbnMgYmVsb3c7IGNoZWNrIFRhYmxlIDEuIgogIGNoZWNrZWQ6IGZhbHNlCiAgdmFsdWVzOgogICAgY29hbDogMi42MCAgICAgICAgIyA2ODcgZ2FsL01XaCwgZ2VuZXJpYyBjb2FsLCBjb29saW5nIHRvd2VyCiAgICBnYXM6IDAuNzUgICAgICAgICAjIDE5OCBnYWwvTVdoLCBjb21iaW5lZCBjeWNsZSwgY29vbGluZyB0b3dlcgogICAgb2lsOiAyLjYwICAgICAgICAgIyBubyBzZXBhcmF0ZSBtZWRpYW47IHRyZWF0ZWQgbGlrZSBzdGVhbSBjb2FsCiAgICBudWNsZWFyOiAyLjU0ICAgICAjIDY3MiBnYWwvTVdoLCBjb29saW5nIHRvd2VyCiAgICBiaW9tYXNzOiAyLjA5ICAgICAjIDU1MyBnYWwvTVdoLCBzdGVhbSwgY29vbGluZyB0b3dlcgogICAgZ2VvdGhlcm1hbDogMS4wMiAgIyAyNzAgZ2FsL01XaCwgYmluYXJ5LCB3ZXQgY29vbGluZzsgZmxhc2ggcGxhbnRzIHVzZSBmYXIgbGVzcwogICAgc29sYXI6IDAuMDA0ICAgICAgIyBQViBwYW5lbCB3YXNoaW5nLCB+MSBnYWwvTVdoCiAgICB3aW5kOiAwLjAKICAgIGh5ZHJvOiAxNy4wICAgICAgICMgcmVzZXJ2b2lyIGV2YXBvcmF0aW9uIChUb3JjZWxsaW5pIGV0IGFsLiAyMDAzLCBOUkVML1RQLTU1MC0zMzkwNSk7IHNlZSBpbmNsdWRlX2h5ZHJvCiAgICBiYXR0ZXJ5OiAwLjAKICAgIHVua25vd246IDAuNzUgICAgICMgdHJlYXRlZCBhcyBnYXMKICBpbmNsdWRlX2h5ZHJvOgogICAgdmFsdWU6IGZhbHNlCiAgICBub3RlOiAiUmVzZXJ2b2lyIGV2YXBvcmF0aW9uIGlzIHNoYXJlZCB3aXRoIGlycmlnYXRpb24sIHN1cHBseSBhbmQgZmxvb2QgY29udHJvbCwgYW5kIGFsbG9jYXRpbmcgaXQgYWxsIHRvIHBvd2VyIGlzIGNvbnRlc3RlZC4gT2ZmIGJ5IGRlZmF1bHQsIHNvIGh5ZHJvLWhlYXZ5IGdyaWRzIGxvb2sgd2F0ZXItbGlnaHQuIFRoaXMgaXMgYSByZWFsIG1vZGVsbGluZyBjaG9pY2U7IHRoZSBibG9nIHNob3VsZCBkaXNjdXNzIGl0LiIKCnJlZ2lvbmFsX3d1ZToKICAjIERpc2Nsb3NlZCBhbm51YWwgb24tc2l0ZSBXVUUgKEwva1doIElUKS4gVXNlZCB0byBjYWxpYnJhdGUgZWFjaCByZWdpb24ncwogICMgY3VydmUgbGV2ZWwuIEFXUyAyMDI0IGZpZ3VyZXMuCiAgc291cmNlOgogICAgdGl0bGU6IEFXUyBXVUUgYW5kIFBVRSBieSByZWdpb24gKHN1c3RhaW5hYmlsaXR5LmFib3V0YW1hem9uLmNvbSwgQ1NWIGFuZCByZWdpb25hbCBmYWN0IHNoZWV0cykKICAgIHVybDogaHR0cHM6Ly9zdXN0YWluYWJpbGl0eS5hYm91dGFtYXpvbi5jb20vYXdzLXd1ZS1wdWUuY3N2CiAgeWVhcjogMjAyNAogIHVuaXQ6IEwva1doCiAgdW5jZXJ0YWludHk6IDAuMTUKICBub3RlOiBWYWx1ZXMgYXMgcmVwb3J0ZWQgYnkgc2VhcmNoIHN1bW1hcmllcyBvZiB0aGUgQVdTIGZhY3Qgc2hlZXRzOyBjaGVjayBhZ2FpbnN0IHRoZSBDU1YuCiAgY2hlY2tlZDogZmFsc2UKICB2YWx1ZXM6CiAgICB1cy1lYXN0LTE6IDAuMTIKICAgIHVzLXdlc3QtMjogMC4xNgogICAgZXUtd2VzdC0xOiAwLjAzCiAgICBldS1ub3J0aC0xOiAwLjAyCiAgICBldS1jZW50cmFsLTE6IDAuMDEKICAgIGFwLXNvdXRoZWFzdC0xOiAxLjY4CiAgICBhc2lhLXBhY2lmaWMtYWdncmVnYXRlOiAwLjk4ICAgIyBBV1MgQXNpYSBQYWNpZmljIGV4Y2x1ZGluZyBDaGluYTsgZmFsbGJhY2sgZm9yIEFQQUMgcmVnaW9ucyB3aXRob3V0IHRoZWlyIG93biBmaWd1cmUKCnB1ZToKICBkZWZhdWx0OgogICAgdmFsdWU6IDEuMTUKICAgIHVuaXQ6IHJhdGlvCiAgICB1bmNlcnRhaW50eTogMC4wNQogICAgc291cmNlOgogICAgICB0aXRsZTogQVdTIGdsb2JhbCBQVUUsIEFtYXpvbiAyMDI0IFN1c3RhaW5hYmlsaXR5IFJlcG9ydAogICAgICB1cmw6IGh0dHBzOi8vc3VzdGFpbmFiaWxpdHkuYWJvdXRhbWF6b24uY29tL3JlcG9ydHMKICAgIG5vdGU6IEZhY2lsaXR5IGVuZXJneSAvIElUIGVuZXJneS4gVXNlZCB3aGVyZSBhIHJlZ2lvbiBoYXMgbm8gZGlzY2xvc2VkIFBVRS4KICAgIGNoZWNrZWQ6IGZhbHNlCgpncHVzOgogICMgQm9hcmQgcG93ZXIgKFREUCkgZnJvbSBOVklESUEgZGF0YSBzaGVldHMsIGluIHdhdHRzLgogIHNvdXJjZToKICAgIHRpdGxlOiBOVklESUEgZGF0YS1jZW50ZXIgR1BVIGRhdGEgc2hlZXRzCiAgICB1cmw6IGh0dHBzOi8vd3d3Lm52aWRpYS5jb20vZW4tdXMvZGF0YS1jZW50ZXIvcHJvZHVjdHMvCiAgdW5pdDogVwogIHVuY2VydGFpbnR5OiAwLjA1CiAgY2hlY2tlZDogZmFsc2UKICB2YWx1ZXM6CiAgICB0NDogNzAKICAgIGw0OiA3MgogICAgdjEwMDogMzAwCiAgICBhMTAwOiA0MDAgICAgICAgIyBTWE0KICAgIGgxMDA6IDcwMCAgICAgICAjIFNYTQoKam9iX2VuZXJneToKICBncHVfdXRpbGlzYXRpb246CiAgICB2YWx1ZTogMC43CiAgICB1bml0OiBmcmFjdGlvbiBvZiBURFAKICAgIHVuY2VydGFpbnR5OiAwLjMKICAgIHNvdXJjZToKICAgICAgdGl0bGU6ICJQYXR0ZXJzb24gZXQgYWwuICgyMDIxKSwgQ2FyYm9uIEVtaXNzaW9ucyBhbmQgTGFyZ2UgTmV1cmFsIE5ldHdvcmsgVHJhaW5pbmcsIGFyWGl2OjIxMDQuMTAzNTAgKG1lYXN1cmVkIGF2ZXJhZ2UgR1BVIHBvd2VyIGJlbG93IFREUCkiCiAgICAgIHVybDogaHR0cHM6Ly9hcnhpdi5vcmcvYWJzLzIxMDQuMTAzNTAKICAgIG5vdGU6IEF2ZXJhZ2UgZHJhdyBhcyBhIHNoYXJlIG9mIFREUCBmb3IgdHJhaW5pbmcuIFJlcGxhY2VkIGJ5IG1lYXN1cmVkIGtXaCAoQ29kZUNhcmJvbikgYWZ0ZXIgYSBqb2IgcnVucy4KICAgIGNoZWNrZWQ6IGZhbHNlCiAgc2VydmVyX292ZXJoZWFkOgogICAgdmFsdWU6IDEuMwogICAgdW5pdDogbXVsdGlwbGllciBvbiBHUFUgZW5lcmd5CiAgICB1bmNlcnRhaW50eTogMC4yCiAgICBzb3VyY2U6CiAgICAgIHRpdGxlOiAiTHVjY2lvbmksIFZpZ3VpZXIsIExpZ296YXQgKDIwMjMpLCBFc3RpbWF0aW5nIHRoZSBDYXJib24gRm9vdHByaW50IG9mIEJMT09NLCBKTUxSIDI0KDI1MykgKG5vbi1HUFUgc2VydmVyIHNoYXJlKSIKICAgICAgdXJsOiBodHRwczovL2ptbHIub3JnL3BhcGVycy92MjQvMjMtMDA2OS5odG1sCiAgICBub3RlOiBDUFUsIG1lbW9yeSwgbmV0d29ya2luZyBhbmQgc3RvcmFnZSBvbiB0aGUgc2FtZSBob3N0LgogICAgY2hlY2tlZDogZmFsc2UKCndhdGVyX3N0cmVzczoKICAjIE11bHRpcGxpZXIgUyhyKSA9IDEgKyBzY29yZSAvIG1heF9zY29yZSwgc28gMS4wIChsb3cgc3RyZXNzKSB0byAyLjAgKGV4dHJlbWVseSBoaWdoKS4KICBtYXhfc2NvcmU6CiAgICB2YWx1ZTogNQogICAgdW5pdDogQXF1ZWR1Y3QgYmFzZWxpbmUgd2F0ZXIgc3RyZXNzIHNjb3JlICgwLTUpCiAgICB1bmNlcnRhaW50eTogMC4wCiAgICBzb3VyY2U6CiAgICAgIHRpdGxlOiBXUkkgQXF1ZWR1Y3QgNC4wIGJhc2VsaW5lIHdhdGVyIHN0cmVzcwogICAgICB1cmw6IGh0dHBzOi8vd3d3LndyaS5vcmcvZGF0YS9hcXVlZHVjdC1nbG9iYWwtbWFwcy00MC1kYXRhCiAgICBub3RlOiBUaGUgc2NhbGluZyB0byAxLTIgaXMgYSBwb2xpY3kgY2hvaWNlLCBub3QgcGh5c2ljcy4gUmVnaW9ucyB3aXRob3V0IGEgc2NvcmUgZ2V0IDEuMC4KICAgIGNoZWNrZWQ6IGZhbHNlCgpjYXJib25faW50ZW5zaXR5OgogIHVuY2VydGFpbnR5OgogICAgdmFsdWU6IDAuMQogICAgdW5pdDogcmVsYXRpdmUKICAgIHVuY2VydGFpbnR5OiAwLjAKICAgIHNvdXJjZToKICAgICAgdGl0bGU6IEVsZWN0cmljaXR5IE1hcHMgbWV0aG9kb2xvZ3kgKGNvbnN1bXB0aW9uLWJhc2VkLCBsaWZlLWN5Y2xlIGNhcmJvbiBpbnRlbnNpdHkpCiAgICAgIHVybDogaHR0cHM6Ly93d3cuZWxlY3RyaWNpdHltYXBzLmNvbS9tZXRob2RvbG9neQogICAgbm90ZTogQXNzdW1lZCBiYW5kIG9uIHByb3ZpZGVyIGNhcmJvbiBpbnRlbnNpdHk7IGZvcmVjYXN0cyBjYXJyeSBtb3JlIChtZWFzdXJlZCBwZXIgcmVnaW9uIGluIFN0ZXAgNCkuCiAgICBjaGVja2VkOiBmYWxzZQoKbWVhc3VyZWRfZW5lcmd5OgogIHVuY2VydGFpbnR5OgogICAgdmFsdWU6IDAuMTUKICAgIHVuaXQ6IHJlbGF0aXZlCiAgICB1bmNlcnRhaW50eTogMC4wCiAgICBzb3VyY2U6CiAgICAgIHRpdGxlOiAiQ29kZUNhcmJvbiBtZXRob2RvbG9neSAoZmFsbHMgYmFjayB0byBURFAtYmFzZWQgZXN0aW1hdGVzIHdpdGhvdXQgUkFQTC9OVk1MIGFjY2VzcywgYXMgb24gRmFyZ2F0ZSkiCiAgICAgIHVybDogaHR0cHM6Ly9tbGNvMi5naXRodWIuaW8vY29kZWNhcmJvbi9tZXRob2RvbG9neS5odG1sCiAgICBub3RlOiBCYW5kIG9uIGVuZXJneSBhZnRlciBhIHJ1bi4gV2l0aG91dCBoYXJkd2FyZSBjb3VudGVycywgQ29kZUNhcmJvbidzIGZpZ3VyZSBpcyBpdHNlbGYgYW4gZXN0aW1hdGUuCiAgICBjaGVja2VkOiBmYWxzZQoKcHJveHlfcnVuOgogICMgVGhlIGRlbW8gImpvYiIgdGhhdCBhY3R1YWxseSBydW5zIGluIHRoZSBjaG9zZW4gcmVnaW9uIGlzIGEgc21hbGwgQ1BVIHRyYWluaW5nIGxvb3AKICAjIChMYW1iZGEgYW5kIEZhcmdhdGUgaGF2ZSBubyBHUFVzKS4gSXRzIGVuZXJneSA9IHZDUFUtaG91cnMgw5cgYXZlcmFnZSB3YXR0cyBwZXIgdkNQVS4KICBtaW5fd2F0dHNfcGVyX3ZjcHU6CiAgICB2YWx1ZTogMC43NAogICAgdW5pdDogVwogICAgdW5jZXJ0YWludHk6IDAuMwogICAgc291cmNlOgogICAgICB0aXRsZTogQ2xvdWQgQ2FyYm9uIEZvb3RwcmludCBtZXRob2RvbG9neSwgQVdTIGF2ZXJhZ2UgbWluIHdhdHRzIHBlciB2Q1BVCiAgICAgIHVybDogaHR0cHM6Ly9naXRodWIuY29tL2Nsb3VkLWNhcmJvbi1mb290cHJpbnQvY2xvdWQtY2FyYm9uLWZvb3RwcmludC9ibG9iL2U0OGM2NTlmNmRhZmM4Yjc4M2U1NzAwNTMwMjRmMjhiODhhYWZjNzkvbWljcm9zaXRlL2RvY3MvTWV0aG9kb2xvZ3kubWQKICAgIG5vdGU6ICJBdmVyYWdlIFdhdHRzID0gTWluICsgdXRpbGlzYXRpb24gw5cgKE1heCAtIE1pbikgKENDRikuIFZhbHVlcyBmcm9tIG1lbW9yeSBvZiB0aGUgQ0NGIHRhYmxlOyBjaGVjay4iCiAgICBjaGVja2VkOiBmYWxzZQogIG1heF93YXR0c19wZXJfdmNwdToKICAgIHZhbHVlOiAzLjUKICAgIHVuaXQ6IFcKICAgIHVuY2VydGFpbnR5OiAwLjMKICAgIHNvdXJjZToKICAgICAgdGl0bGU6IENsb3VkIENhcmJvbiBGb290cHJpbnQgbWV0aG9kb2xvZ3ksIEFXUyBhdmVyYWdlIG1heCB3YXR0cyBwZXIgdkNQVQogICAgICB1cmw6IGh0dHBzOi8vZ2l0aHViLmNvbS9jbG91ZC1jYXJib24tZm9vdHByaW50L2Nsb3VkLWNhcmJvbi1mb290cHJpbnQvYmxvYi9lNDhjNjU5ZjZkYWZjOGI3ODNlNTcwMDUzMDI0ZjI4Yjg4YWFmYzc5L21pY3Jvc2l0ZS9kb2NzL01ldGhvZG9sb2d5Lm1kCiAgICBub3RlOiBUaGUgcHJveHkgbG9vcCBydW5zIG9uZSB2Q1BVIGF0IH4xMDAlLCBzbyBpdCBzaXRzIG5lYXIgbWF4LgogICAgY2hlY2tlZDogZmFsc2UK",
"model/regions.py": "IiIiUmVnaW9uIGNvbmZpZ3VyYXRpb24gZnJvbSBkYXRhL3JlZ2lvbnMueWFtbC4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKZnJvbSBmdW5jdG9vbHMgaW1wb3J0IGxydV9jYWNoZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCB5YW1sCgpmcm9tIC4gaW1wb3J0IGNvZWZmaWNpZW50cyBhcyBjCgpQQVRIID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudC5wYXJlbnQgLyAiZGF0YSIgLyAicmVnaW9ucy55YW1sIgoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIFJlZ2lvbjoKICAgIGlkOiBzdHIKICAgIG5hbWU6IHN0cgogICAgZ2VvOiBzdHIKICAgIGxhdDogZmxvYXQKICAgIGxvbjogZmxvYXQKICAgIGNvb2xpbmdfdHlwZTogc3RyCiAgICBlbGVjdHJpY2l0eV9tYXBzX3pvbmU6IHN0ciB8IE5vbmUKICAgIHB1ZTogZmxvYXQKICAgIGRpc2Nsb3NlZF93dWU6IGZsb2F0IHwgTm9uZQogICAgd3VlX3NjYWxlOiBmbG9hdCAgICAgICAgICAgICAgIyBjYWxpYnJhdGlvbiBmYWN0b3I7IDEuMCB1bnRpbCBjYWxpYnJhdGVkIGZyb20gd2VhdGhlciBoaXN0b3J5CiAgICB3YXRlcl9zdHJlc3Nfc2NvcmU6IGZsb2F0IHwgTm9uZQogICAgZ3JpZF9taXg6IGRpY3QgPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKICAgIHR5cGljYWxfY2lfZ19wZXJfa3doOiBmbG9hdCB8IE5vbmUgPSBOb25lCgogICAgQHByb3BlcnR5CiAgICBkZWYgc3RyZXNzX211bHRpcGxpZXIoc2VsZikgLT4gZmxvYXQ6CiAgICAgICAgaWYgc2VsZi53YXRlcl9zdHJlc3Nfc2NvcmUgaXMgTm9uZToKICAgICAgICAgICAgcmV0dXJuIDEuMAogICAgICAgIHJldHVybiAxLjAgKyBzZWxmLndhdGVyX3N0cmVzc19zY29yZSAvIGMudmFsdWUoIndhdGVyX3N0cmVzcyIsICJtYXhfc2NvcmUiKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGNhbGlicmF0ZWQoc2VsZikgLT4gYm9vbDoKICAgICAgICByZXR1cm4gc2VsZi53dWVfc2NhbGUgIT0gMS4wIG9yIHNlbGYuY29vbGluZ190eXBlID09ICJhaXIiCgoKQGxydV9jYWNoZShtYXhzaXplPTEpCmRlZiBsb2FkKHBhdGg6IFBhdGggPSBQQVRIKSAtPiBkaWN0W3N0ciwgUmVnaW9uXToKICAgIHJhdyA9IHlhbWwuc2FmZV9sb2FkKHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAgZGlzY2xvc2VkID0gYy5sb2FkKClbInJlZ2lvbmFsX3d1ZSJdWyJ2YWx1ZXMiXQogICAgZGVmYXVsdF9wdWUgPSBjLnZhbHVlKCJwdWUiLCAiZGVmYXVsdCIpCiAgICBvdXQgPSB7fQogICAgZm9yIHIgaW4gcmF3WyJyZWdpb25zIl06CiAgICAgICAgd3VlX2tleSA9IHIuZ2V0KCJkaXNjbG9zZWRfd3VlX2tleSIsIHJbImlkIl0pCiAgICAgICAgb3V0W3JbImlkIl1dID0gUmVnaW9uKAogICAgICAgICAgICBpZD1yWyJpZCJdLAogICAgICAgICAgICBuYW1lPXJbIm5hbWUiXSwKICAgICAgICAgICAgZ2VvPXJbImdlbyJdLAogICAgICAgICAgICBsYXQ9clsibGF0Il0sCiAgICAgICAgICAgIGxvbj1yWyJsb24iXSwKICAgICAgICAgICAgY29vbGluZ190eXBlPXJbImNvb2xpbmdfdHlwZSJdLAogICAgICAgICAgICBlbGVjdHJpY2l0eV9tYXBzX3pvbmU9ci5nZXQoImVsZWN0cmljaXR5X21hcHNfem9uZSIpLAogICAgICAgICAgICBwdWU9ci5nZXQoInB1ZSIpIG9yIGRlZmF1bHRfcHVlLAogICAgICAgICAgICBkaXNjbG9zZWRfd3VlPWRpc2Nsb3NlZC5nZXQod3VlX2tleSksCiAgICAgICAgICAgIHd1ZV9zY2FsZT1yLmdldCgid3VlX3NjYWxlIikgb3IgMS4wLAogICAgICAgICAgICB3YXRlcl9zdHJlc3Nfc2NvcmU9ci5nZXQoIndhdGVyX3N0cmVzc19zY29yZSIpLAogICAgICAgICAgICBncmlkX21peD1yLmdldCgiZ3JpZF9taXgiLCB7fSkuZ2V0KCJzaGFyZXMiLCB7fSksCiAgICAgICAgICAgIHR5cGljYWxfY2lfZ19wZXJfa3doPXIuZ2V0KCJ0eXBpY2FsX2NpX2dfcGVyX2t3aCIpLAogICAgICAgICkKICAgIHJldHVybiBvdXQKCgpkZWYgZ2V0KHJlZ2lvbl9pZDogc3RyKSAtPiBSZWdpb246CiAgICByZWdpb25zID0gbG9hZCgpCiAgICB0cnk6CiAgICAgICAgcmV0dXJuIHJlZ2lvbnNbcmVnaW9uX2lkXQogICAgZXhjZXB0IEtleUVycm9yOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ1bmtub3duIHJlZ2lvbiB7cmVnaW9uX2lkIXJ9OyBrbm93bjogeycsICcuam9pbihyZWdpb25zKX0iKSBmcm9tIE5vbmUK",
"model/wetbulb.py": "IiIiV2V0LWJ1bGIgdGVtcGVyYXR1cmUuCgpgYHdldF9idWxiKHQsIHJoLCBwX2hwYSlgYCBzb2x2ZXMgdGhlIHBzeWNocm9tZXRyaWMgZXF1YXRpb24gKFdNTy1Oby4gOCk6CgogICAgZSA9IGVfdyhUdykgLSBBIMK3ICgxICsgMC4wMDA5NDTCt1R3KSDCtyBQIMK3IChUIC0gVHcpCgpmb3IgVHcgYnkgYmlzZWN0aW9uLCB3aGVyZSBlID0gUkggwrcgZV93KFQpIGFuZCBlX3cgaXMgdGhlIE1hZ251cyBzYXR1cmF0aW9uCnZhcG91ciBwcmVzc3VyZSBvdmVyIHdhdGVyIChlX3cgPSA2LjExMsK3ZXhwKDE3LjYywrdUIC8gKDI0My4xMiArIFQpKSBoUGEsCldNTy1Oby4gOCkuIFRoaXMgdXNlcyBzdXJmYWNlIHByZXNzdXJlLCBzbyBpdCBpcyBjb3JyZWN0IGF0IGFsdGl0dWRlLgoKYGB3ZXRfYnVsYl9zdHVsbCh0LCByaClgYCBpcyB0aGUgY2xvc2VkLWZvcm0gU3R1bGwgKDIwMTEpIGZpdC4gSXQgYXNzdW1lcwpzZWEtbGV2ZWwgcHJlc3N1cmUgKDEwMTMuMjUgaFBhKTsgU3R1bGwgcmVwb3J0cyBlcnJvcnMgYmV0d2VlbiAtMSBhbmQgKzAuNjUgwrBDCmZvciBSSCA1LTk5JSBhbmQgLTIwLTUwIMKwQy4gSXQgaXMga2VwdCBhcyBhIGNyb3NzLWNoZWNrOiB0aGUgdHdvIGFncmVlIG5lYXIgc2VhIGxldmVsLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBtYXRoCgpmcm9tIC4gaW1wb3J0IGNvZWZmaWNpZW50cwoKU0VBX0xFVkVMX0hQQSA9IDEwMTMuMjUKCgpkZWYgc2F0dXJhdGlvbl92YXBvdXJfcHJlc3N1cmVfaHBhKHQ6IGZsb2F0KSAtPiBmbG9hdDoKICAgICIiIk1hZ251cyBmb3JtdWxhIG92ZXIgd2F0ZXIsIGhQYSAoV01PLU5vLiA4KS4iIiIKICAgIHJldHVybiA2LjExMiAqIG1hdGguZXhwKDE3LjYyICogdCAvICgyNDMuMTIgKyB0KSkKCgpkZWYgd2V0X2J1bGIodDogZmxvYXQsIHJoOiBmbG9hdCwgcF9ocGE6IGZsb2F0ID0gU0VBX0xFVkVMX0hQQSkgLT4gZmxvYXQ6CiAgICAiIiJQc3ljaHJvbWV0cmljIHdldC1idWxiIHRlbXBlcmF0dXJlICjCsEMpIGZyb20gYWlyIHRlbXBlcmF0dXJlICjCsEMpLCBSSCAoJSkgYW5kIHByZXNzdXJlIChoUGEpLiIiIgogICAgaWYgbm90IDAgPCByaCA8PSAxMDA6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInJlbGF0aXZlIGh1bWlkaXR5IG11c3QgYmUgaW4gKDAsIDEwMF0sIGdvdCB7cmh9IikKICAgIGEgPSBjb2VmZmljaWVudHMudmFsdWUoInBoeXNpY3MiLCAicHN5Y2hyb21ldGVyX2NvZWZmaWNpZW50IikKICAgIGUgPSByaCAvIDEwMC4wICogc2F0dXJhdGlvbl92YXBvdXJfcHJlc3N1cmVfaHBhKHQpCgogICAgZGVmIHJlc2lkdWFsKHR3OiBmbG9hdCkgLT4gZmxvYXQ6CiAgICAgICAgcmV0dXJuIHNhdHVyYXRpb25fdmFwb3VyX3ByZXNzdXJlX2hwYSh0dykgLSBhICogKDEgKyAwLjAwMDk0NCAqIHR3KSAqIHBfaHBhICogKHQgLSB0dykgLSBlCgogICAgbG8sIGhpID0gdCAtIDYwLjAsIHQgICMgcmVzaWR1YWwobG8pIDwgMCA8IHJlc2lkdWFsKGhpKSBmb3IgYW55IHBoeXNpY2FsIGlucHV0CiAgICBmb3IgXyBpbiByYW5nZSg2MCk6CiAgICAgICAgbWlkID0gKGxvICsgaGkpIC8gMgogICAgICAgIGlmIHJlc2lkdWFsKG1pZCkgPiAwOgogICAgICAgICAgICBoaSA9IG1pZAogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGxvID0gbWlkCiAgICByZXR1cm4gKGxvICsgaGkpIC8gMgoKCmRlZiB3ZXRfYnVsYl9zdHVsbCh0OiBmbG9hdCwgcmg6IGZsb2F0KSAtPiBmbG9hdDoKICAgICIiIlN0dWxsICgyMDExKSwgSi4gQXBwbC4gTWV0ZW9yLiBDbGltYXRvbC4gNTA6MjI2Ny4gU2VhLWxldmVsIHByZXNzdXJlIG9ubHkuIiIiCiAgICByaCA9IG1pbihtYXgocmgsIDUuMCksIDk5LjApCiAgICByZXR1cm4gKAogICAgICAgIHQgKiBtYXRoLmF0YW4oMC4xNTE5NzcgKiBtYXRoLnNxcnQocmggKyA4LjMxMzY1OSkpCiAgICAgICAgKyBtYXRoLmF0YW4odCArIHJoKQogICAgICAgIC0gbWF0aC5hdGFuKHJoIC0gMS42NzYzMzEpCiAgICAgICAgKyAwLjAwMzkxODM4ICogcmgqKjEuNSAqIG1hdGguYXRhbigwLjAyMzEwMSAqIHJoKQogICAgICAgIC0gNC42ODYwMzUKICAgICkK",
"forecasting/__init__.py": "IiIiU3RlcCA0OiA0OCBoIGZvcmVjYXN0IG1vZGVscyBmb3Igd2V0LWJ1bGIgYW5kIGNhcmJvbiBpbnRlbnNpdHksIHdpdGggYmFzZWxpbmVzLgoKVHJhaW5pbmcgKHNjaWtpdC1sZWFybikgaGFwcGVucyBvZmZsaW5lIGluIHNjcmlwdHMvdHJhaW5fZm9yZWNhc3RzLnB5LiBUaGUgdHJhaW5lZApncmFkaWVudC1ib29zdGVkIHRyZWVzIGFyZSBleHBvcnRlZCB0byBKU09OIChmb3JlY2FzdGluZy90cmVlcy5weSkgYW5kIGV2YWx1YXRlZCBpbgpwdXJlIFB5dGhvbiwgc28gdGhlIExhbWJkYSBsYXllciBuZWVkcyBuZWl0aGVyIHNjaWtpdC1sZWFybiBub3IgbnVtcHkuCiIiIgo=",
"forecasting/evaluate.py": "IiIiVHJhaW4gYW5kIGp1ZGdlIG9uZSBtb2RlbCBhZ2FpbnN0IGl0cyBiYXNlbGluZXMgb24gaGVsZC1vdXQgZGF5cy4KClRoZSBydWxlIGZyb20gdGhlIHBsYW46IGEgbW9kZWwgaXMgb25seSB1c2VkIGlmIGl0cyBoZWxkLW91dCBNQUUgYmVhdHMgQk9USApwZXJzaXN0ZW5jZSBhbmQgdGhlIHJhdyBwcm92aWRlciBmb3JlY2FzdCAoZm9yIGNhcmJvbiwgd2hlcmUgdGhlIGZyZWUgdGllciBoYXMKbm8gcHJvdmlkZXIgZm9yZWNhc3QsIHRoZSBwcm92aWRlciBiYXNlbGluZSBpcyAibGF0ZXN0IHZhbHVlIGhlbGQiKS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCgpAZGF0YWNsYXNzCmNsYXNzIFZlcmRpY3Q6CiAgICB0YXJnZXQ6IHN0cgogICAgcmVnaW9uOiBzdHIKICAgIG5fdHJhaW46IGludAogICAgbl90ZXN0OiBpbnQKICAgIG1hZV9tb2RlbDogZmxvYXQKICAgIG1hZV9wZXJzaXN0ZW5jZTogZmxvYXQKICAgIG1hZV9wcm92aWRlcjogZmxvYXQKICAgIHVzZV9tb2RlbDogYm9vbAoKICAgIGRlZiBhc19kaWN0KHNlbGYpIC0+IGRpY3Q6CiAgICAgICAgcmV0dXJuIHtrOiAocm91bmQodiwgNCkgaWYgaXNpbnN0YW5jZSh2LCBmbG9hdCkgZWxzZSB2KSBmb3IgaywgdiBpbiBzZWxmLl9fZGljdF9fLml0ZW1zKCl9CgoKZGVmIG1hZShhLCBiKSAtPiBmbG9hdDoKICAgIHJldHVybiBzdW0oYWJzKHggLSB5KSBmb3IgeCwgeSBpbiB6aXAoYSwgYikpIC8gbWF4KDEsIGxlbihhKSkKCgpkZWYgZml0X2FuZF9qdWRnZSh0YXJnZXQ6IHN0ciwgcmVnaW9uOiBzdHIsIHJvd3M6IGxpc3RbZGljdF0sIHk6IGxpc3RbZmxvYXRdLCBmZWF0dXJlczogbGlzdFtzdHJdLAogICAgICAgICAgICAgICAgICBwZXJzaXN0ZW5jZTogbGlzdFtmbG9hdF0sIHByb3ZpZGVyOiBsaXN0W2Zsb2F0XSwgdGVzdF9tYXNrOiBsaXN0W2Jvb2xdLCBzZWVkOiBpbnQgPSAwKToKICAgICIiIlJldHVybnMgKFZlcmRpY3QsIGV4cG9ydGVkIG1vZGVsIGRpY3Qgb3IgTm9uZSkuIiIiCiAgICBmcm9tIHNrbGVhcm4uZW5zZW1ibGUgaW1wb3J0IEdyYWRpZW50Qm9vc3RpbmdSZWdyZXNzb3IKCiAgICBmcm9tIC50cmVlcyBpbXBvcnQgZXhwb3J0LCBwcmVkaWN0CgogICAgdHJhaW4gPSBbaSBmb3IgaSwgdCBpbiBlbnVtZXJhdGUodGVzdF9tYXNrKSBpZiBub3QgdF0KICAgIHRlc3QgPSBbaSBmb3IgaSwgdCBpbiBlbnVtZXJhdGUodGVzdF9tYXNrKSBpZiB0XQogICAgaWYgbGVuKHRyYWluKSA8IDQ4IG9yIGxlbih0ZXN0KSA8IDI0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7cmVnaW9ufS97dGFyZ2V0fTogbm90IGVub3VnaCBkYXRhICh0cmFpbiB7bGVuKHRyYWluKX0sIHRlc3Qge2xlbih0ZXN0KX0pIikKICAgIFggPSBbW3JbZl0gZm9yIGYgaW4gZmVhdHVyZXNdIGZvciByIGluIHJvd3NdCiAgICBtb2RlbCA9IEdyYWRpZW50Qm9vc3RpbmdSZWdyZXNzb3Iobl9lc3RpbWF0b3JzPTE1MCwgbWF4X2RlcHRoPTMsIGxlYXJuaW5nX3JhdGU9MC4wNSwgc3Vic2FtcGxlPTAuOCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICByYW5kb21fc3RhdGU9c2VlZCkKICAgIG1vZGVsLmZpdChbWFtpXSBmb3IgaSBpbiB0cmFpbl0sIFt5W2ldIGZvciBpIGluIHRyYWluXSkKICAgIGV4cG9ydGVkID0gZXhwb3J0KG1vZGVsLCBmZWF0dXJlcykKICAgIHByZWQgPSBwcmVkaWN0KGV4cG9ydGVkLCBbcm93c1tpXSBmb3IgaSBpbiB0ZXN0XSkKICAgIHl0ID0gW3lbaV0gZm9yIGkgaW4gdGVzdF0KICAgIHYgPSBWZXJkaWN0KHRhcmdldCwgcmVnaW9uLCBsZW4odHJhaW4pLCBsZW4odGVzdCksIG1hZShwcmVkLCB5dCksCiAgICAgICAgICAgICAgICBtYWUoW3BlcnNpc3RlbmNlW2ldIGZvciBpIGluIHRlc3RdLCB5dCksIG1hZShbcHJvdmlkZXJbaV0gZm9yIGkgaW4gdGVzdF0sIHl0KSwgRmFsc2UpCiAgICB2LnVzZV9tb2RlbCA9IHYubWFlX21vZGVsIDwgdi5tYWVfcGVyc2lzdGVuY2UgYW5kIHYubWFlX21vZGVsIDwgdi5tYWVfcHJvdmlkZXIKICAgIHJldHVybiB2LCBleHBvcnRlZAo=",
"forecasting/features.py": "IiIiRmVhdHVyZSByb3dzIGZvciB0aGUgdHdvIG1vZGVscy4gU2FtZSBjb2RlIGF0IHRyYWluaW5nIGFuZCBzZXJ2aW5nIHRpbWUuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbWF0aApmcm9tIGRhdGV0aW1lIGltcG9ydCBkYXRldGltZQoKV0VUQlVMQl9GRUFUVVJFUyA9IFsicHJvdmlkZXJfdF93YiIsICJwcm92aWRlcl90X2RiIiwgInByb3ZpZGVyX3JoIiwgImxlYWRfaCIsICJob3VyX3NpbiIsICJob3VyX2NvcyIsCiAgICAgICAgICAgICAgICAgICAgImRveV9zaW4iLCAiZG95X2NvcyIsICJyZWNlbnRfcmVzaWR1YWwiXQpDQVJCT05fRkVBVFVSRVMgPSBbImxhc3RfY2kiLCAibGVhZF9oIiwgImhvdXJfc2luIiwgImhvdXJfY29zIiwgImRvdyIsICJzYW1lX2hvdXJfeWVzdGVyZGF5Il0KCgpkZWYgX3RpbWUoaG91cjogc3RyKSAtPiBkaWN0OgogICAgZHQgPSBkYXRldGltZS5zdHJwdGltZShob3VyWzoxM10sICIlWS0lbS0lZFQlSCIpCiAgICBoID0gMiAqIG1hdGgucGkgKiBkdC5ob3VyIC8gMjQKICAgIGQgPSAyICogbWF0aC5waSAqIGR0LnRpbWV0dXBsZSgpLnRtX3lkYXkgLyAzNjUuMjUKICAgIHJldHVybiB7ImhvdXJfc2luIjogbWF0aC5zaW4oaCksICJob3VyX2NvcyI6IG1hdGguY29zKGgpLCAiZG95X3NpbiI6IG1hdGguc2luKGQpLCAiZG95X2NvcyI6IG1hdGguY29zKGQpLAogICAgICAgICAgICAiZG93IjogZHQud2Vla2RheSgpfQoKCmRlZiB3ZXRidWxiX3Jvdyhob3VyOiBzdHIsIGxlYWRfaDogZmxvYXQsIHByb3ZpZGVyX3Rfd2I6IGZsb2F0LCBwcm92aWRlcl90X2RiOiBmbG9hdCwgcHJvdmlkZXJfcmg6IGZsb2F0LAogICAgICAgICAgICAgICAgcmVjZW50X3Jlc2lkdWFsOiBmbG9hdCkgLT4gZGljdDoKICAgICIiInJlY2VudF9yZXNpZHVhbDogb2JzZXJ2ZWQgbWludXMgcHJvdmlkZXIgd2V0LWJ1bGIgYXQgaXNzdWUgdGltZSAoMCBpZiB1bmtub3duKS4iIiIKICAgIHJldHVybiB7InByb3ZpZGVyX3Rfd2IiOiBwcm92aWRlcl90X3diLCAicHJvdmlkZXJfdF9kYiI6IHByb3ZpZGVyX3RfZGIsICJwcm92aWRlcl9yaCI6IHByb3ZpZGVyX3JoLAogICAgICAgICAgICAibGVhZF9oIjogbGVhZF9oLCAicmVjZW50X3Jlc2lkdWFsIjogcmVjZW50X3Jlc2lkdWFsLCAqKl90aW1lKGhvdXIpfQoKCmRlZiBjYXJib25fcm93KGhvdXI6IHN0ciwgbGVhZF9oOiBmbG9hdCwgbGFzdF9jaTogZmxvYXQsIHNhbWVfaG91cl95ZXN0ZXJkYXk6IGZsb2F0IHwgTm9uZSkgLT4gZGljdDoKICAgIHJldHVybiB7Imxhc3RfY2kiOiBsYXN0X2NpLCAibGVhZF9oIjogbGVhZF9oLAogICAgICAgICAgICAic2FtZV9ob3VyX3llc3RlcmRheSI6IGxhc3RfY2kgaWYgc2FtZV9ob3VyX3llc3RlcmRheSBpcyBOb25lIGVsc2Ugc2FtZV9ob3VyX3llc3RlcmRheSwgKipfdGltZShob3VyKX0K",
"forecasting/trees.py": "IiIiRXhwb3J0IGEgZml0dGVkIHNrbGVhcm4gR3JhZGllbnRCb29zdGluZ1JlZ3Jlc3NvciB0byBKU09OIGFuZCBwcmVkaWN0IHdpdGhvdXQgc2tsZWFybi4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCgpkZWYgZXhwb3J0KG1vZGVsLCBmZWF0dXJlX25hbWVzOiBsaXN0W3N0cl0pIC0+IGRpY3Q6CiAgICAiIiJTZXJpYWxpc2UgYSBmaXR0ZWQgc2tsZWFybi5lbnNlbWJsZS5HcmFkaWVudEJvb3N0aW5nUmVncmVzc29yIChsb3NzPSdzcXVhcmVkX2Vycm9yJykuIiIiCiAgICB0cmVlcyA9IFtdCiAgICBmb3IgZXN0IGluIG1vZGVsLmVzdGltYXRvcnNfWzosIDBdOgogICAgICAgIHQgPSBlc3QudHJlZV8KICAgICAgICB0cmVlcy5hcHBlbmQoewogICAgICAgICAgICAiZmVhdHVyZSI6IHQuZmVhdHVyZS50b2xpc3QoKSwKICAgICAgICAgICAgInRocmVzaG9sZCI6IFtyb3VuZChmbG9hdCh4KSwgNikgZm9yIHggaW4gdC50aHJlc2hvbGRdLAogICAgICAgICAgICAibGVmdCI6IHQuY2hpbGRyZW5fbGVmdC50b2xpc3QoKSwKICAgICAgICAgICAgInJpZ2h0IjogdC5jaGlsZHJlbl9yaWdodC50b2xpc3QoKSwKICAgICAgICAgICAgInZhbHVlIjogW3JvdW5kKGZsb2F0KHZbMF1bMF0pLCA2KSBmb3IgdiBpbiB0LnZhbHVlXSwKICAgICAgICB9KQogICAgaW5pdCA9IG1vZGVsLmluaXRfCiAgICBiYXNlID0gZmxvYXQoaW5pdC5jb25zdGFudF9bMF1bMF0pIGlmIGhhc2F0dHIoaW5pdCwgImNvbnN0YW50XyIpIGVsc2UgZmxvYXQobW9kZWwuX3Jhd19wcmVkaWN0X2luaXQoW1swXSAqIGxlbihmZWF0dXJlX25hbWVzKV0pWzBdWzBdKQogICAgcmV0dXJuIHsia2luZCI6ICJnYnIiLCAiZmVhdHVyZXMiOiBmZWF0dXJlX25hbWVzLCAibGVhcm5pbmdfcmF0ZSI6IGZsb2F0KG1vZGVsLmxlYXJuaW5nX3JhdGUpLAogICAgICAgICAgICAiYmFzZSI6IHJvdW5kKGJhc2UsIDYpLCAidHJlZXMiOiB0cmVlc30KCgpkZWYgcHJlZGljdF9vbmUobW9kZWw6IGRpY3QsIHg6IGxpc3RbZmxvYXRdKSAtPiBmbG9hdDoKICAgIHRvdGFsID0gbW9kZWxbImJhc2UiXQogICAgbHIgPSBtb2RlbFsibGVhcm5pbmdfcmF0ZSJdCiAgICBmb3IgdCBpbiBtb2RlbFsidHJlZXMiXToKICAgICAgICBub2RlID0gMAogICAgICAgIHdoaWxlIHRbImxlZnQiXVtub2RlXSAhPSAtMToKICAgICAgICAgICAgbm9kZSA9IHRbImxlZnQiXVtub2RlXSBpZiB4W3RbImZlYXR1cmUiXVtub2RlXV0gPD0gdFsidGhyZXNob2xkIl1bbm9kZV0gZWxzZSB0WyJyaWdodCJdW25vZGVdCiAgICAgICAgdG90YWwgKz0gbHIgKiB0WyJ2YWx1ZSJdW25vZGVdCiAgICByZXR1cm4gdG90YWwKCgpkZWYgcHJlZGljdChtb2RlbDogZGljdCwgcm93czogbGlzdFtkaWN0XSkgLT4gbGlzdFtmbG9hdF06CiAgICBuYW1lcyA9IG1vZGVsWyJmZWF0dXJlcyJdCiAgICByZXR1cm4gW3ByZWRpY3Rfb25lKG1vZGVsLCBbZmxvYXQocltuXSkgZm9yIG4gaW4gbmFtZXNdKSBmb3IgciBpbiByb3dzXQo="
}

ROOT = pathlib.Path('/content/pravaah')
for rel, data in FILES.items():
    path = ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(base64.b64decode(data))
print(f'unpacked {len(FILES)} files to {ROOT}')

## Settings
`DAYS` is how much weather history to train on. Leave `REGIONS` empty for all 8 regions, or list some,
e.g. `'eu-north-1,ap-south-1'`, to test quickly.

In [ ]:
DAYS = 60
REGIONS = ''   # '' = all regions

## Optional: carbon history
Skip this cell unless you have synced the history from S3
(`aws s3 sync s3://<BucketName>/history data/history`). Upload the `<region>.csv` files here and carbon
models get trained too. Without them, carbon is reported as *skipped: not enough real history yet*.

In [ ]:
import pathlib
try:
    from google.colab import files
    up = files.upload()
    hist = pathlib.Path('/content/pravaah/data/history'); hist.mkdir(parents=True, exist_ok=True)
    for name, data in up.items():
        (hist / name).write_bytes(data)
    print('saved', sorted(up))
except ImportError:
    print('not running in Colab; copy your CSVs into /content/pravaah/data/history yourself')

## Train and judge

In [ ]:
import subprocess, sys
cmd = [sys.executable, 'scripts/train_forecasts.py', '--days', str(DAYS)] + (['--regions', REGIONS] if REGIONS else [])
run = subprocess.run(cmd, cwd='/content/pravaah', capture_output=True, text=True)
print(run.stdout)
if run.returncode:
    print(run.stderr[-2000:])
    raise SystemExit('training failed (see above). A 429 means the Open-Meteo daily quota: retry after 00:00 UTC.')

## Verdicts
`use_model = True` means the model beat **both** baselines on the held-out week and will be shipped.
Quote this table honestly in the blog and video, including the models that were **not** used.

In [ ]:
import json, pandas as pd
m = json.load(open('/content/pravaah/forecasting/models/metrics.json'))
print('trained at', m['trained_at'], '| held-out days:', m['test_days'])
rows = [r for r in m['results']]
df = pd.DataFrame(rows)
cols = [c for c in ('region', 'target', 'n_train', 'n_test', 'mae_model', 'mae_persistence', 'mae_provider', 'use_model', 'skipped') if c in df]
df[cols]

## Download

In [ ]:
import pathlib, shutil
models = pathlib.Path('/content/pravaah/forecasting/models')
print('models kept:', sorted(p.name for p in models.glob('*-*.json')) or 'none (no model beat both baselines)')
shutil.make_archive('/content/forecast-models', 'zip', models)
try:
    from google.colab import files
    files.download('/content/forecast-models.zip')
except ImportError:
    print('zip at /content/forecast-models.zip')

## Next
1. Unzip into the repo's `forecasting/models/` (replace what is there; delete model files that are not in the zip).
2. Commit them and open a PR.
3. Redeploy so the Lambda picks them up (`make build`, then `sam deploy`). The `predict` step uses only files
   whose verdict says `use_model: true`.